# Data Extraction Evaluation Against PERG Ground Truth

## Overview

This notebook evaluates the data extraction performance of AgentSLR (AI4EpiReview) against human-annotated ground truth data from the PERG (Priority Pathogen Epidemiology Review Group) project. The evaluation covers three extraction types:

1. **Models** - Transmission model extraction from epidemiological literature
2. **Outbreaks** - Outbreak event extraction with temporal and geographic information
3. **Parameters** - Epidemiological parameter extraction including values and uncertainties

The evaluation framework measures performance across three dimensions:
- **Flagging** - Whether the system correctly identifies relevant data types per article
- **Count** - Whether the volume of extractions matches the ground truth
- **Extraction** - Field-level accuracy of matched extraction pairs

## Methodology

### Schema Validation and Data Quality

Prior to evaluation, we validate and filter ground truth extractions to ensure that only properly formatted annotations are compared. For fields typed as `Enum` values in the extraction schemas, we define acceptable values based on the PERG REDCap survey schema, which standardises entries through dropdown lists. Multi-select fields are handled as `List[Enum]` types in the tool call schemas.

We filter any ground truth extractions where `Enum`-typed values do not accord with the schema, to avoid penalising AI4Epi for extractions it is not permitted to produce. Because of schema verification applied in the tool-calling stage, AI4Epi produces no such invalid extractions.

After validation, we align datasets at the article level using shared identifiers, retaining only articles present in both datasets.

### Evaluation Framework

We evaluate extraction performance according to three measures: **Flagging**, **Count**, and **Extraction**. All three are operationalised with standard classification metrics (precision and recall).

#### Flagging

Flagging measures whether AI4Epi correctly identifies the relevant data types to extract from each article. This considers all (article, data_type) pairs:

$$y(\langle\mathrm{article}, \mathrm{data\_type}\rangle) = \begin{cases} 1 & \text{if human extraction exists} \\ 0 & \text{otherwise} \end{cases}$$

$$\hat{y}(\langle\mathrm{article}, \mathrm{data\_type}\rangle) = \begin{cases} 1 & \text{if AI4Epi identifies data type as relevant} \\ 0 & \text{otherwise} \end{cases}$$

#### Count

Count measures whether the volume of AI4Epi extractions accords with ground truth. We use a partial credit scheme: if an article has $n$ items in the reference and the extractor identifies $\hat{n}$ items:

$$\mathrm{TP} = \min(n, \hat{n})$$
$$\mathrm{FP} = \max(0, \hat{n} - n)$$
$$\mathrm{FN} = \max(0, n - \hat{n})$$

#### Extraction

Extraction evaluation faces a matching challenge: extractions lack unique identifiers for canonical correspondence. To assess field-level quality, we establish optimal one-to-one correspondences between ground truth and AI4Epi extractions within each article by computing pairwise similarity.

For each extraction pair, we compare key fields $\mathcal{F}$ using normalised weights. The similarity between a true extraction $E$ and an AI4Epi extraction $\hat{E}$ is:

$$s(E, \hat{E}) = \sum_{k \in \mathcal{F}} w_k \cdot d_k(E[k], \hat{E}[k])$$

where $w_k$ is the normalised weight for field $k$ (with $\sum_{k \in \mathcal{F}} w_k = 1$) and $d_k$ is a distance metric:

$$d_k(v, \hat{v}) = \begin{cases} 1 & \text{if } v \text{ takes a single value and } v = \hat{v} \\ 0 & \text{if } v \text{ takes a single value and } v \neq \hat{v} \\ J(v, \hat{v}) = \frac{|v \cap \hat{v}|}{|v \cup \hat{v}|} & \text{if } v \text{ takes a set of values} \end{cases}$$

We apply the modified Jonker-Volgenant algorithm using `scipy.optimise.linear_sum_assignment()` to the cost matrix (cost = 1 - s), finding the matching that maximises total similarity.

## Setup and Imports

In [1]:
import os
import re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from tqdm import tqdm

root_path = "../"
EVALS_BASE_DIR = Path(root_path) / "data" / "agentslr" / "evals" / "data_extraction"
EVALS_MODELS_DIR = EVALS_BASE_DIR / "models"
EVALS_OUTBREAKS_DIR = EVALS_BASE_DIR / "outbreaks"
EVALS_PARAMETERS_DIR = EVALS_BASE_DIR / "parameters"

os.makedirs(EVALS_MODELS_DIR, exist_ok=True)
os.makedirs(EVALS_OUTBREAKS_DIR, exist_ok=True)
os.makedirs(EVALS_PARAMETERS_DIR, exist_ok=True)

if root_path not in sys.path:
    sys.path.insert(0, root_path)

from utils.evals.create_perg_conditioned_files import add_perg_columns_from_screening
from utils.schemas import (
    METHOD_MAPPING,
    PAIRED_UNCERTAINTY_MAPPING,
    PARAMETER_CLASSES_MAPPING,
    PARAMETER_UNITS_MAPPING,
    POPULATION_GROUP_MAPPING,
    POPULATION_SAMPLE_TYPE_MAPPING,
    POPULATION_SEX_MAPPING,
    SINGLE_TYPE_UNCERTAINTY_MAPPING,
    STATISTICAL_APPROACH_MAPPING,
    VALUE_TYPE_MAPPING,
)

warnings.filterwarnings("ignore", category=FutureWarning)

## CSV Utilities

In [25]:
def _f1_from_pr(p, r):
    if pd.isna(p) or pd.isna(r):
        return np.nan
    denom = p + r
    return 0.0 if denom == 0 else (2 * p * r) / denom


def save_detailed_metrics_csv(df, component_name, output_dir=None, filename_suffix="detailed"):
    output_dir = Path(output_dir) if output_dir else EVALS_BASE_DIR / component_name
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / f"extraction_{component_name}_{filename_suffix}.csv"
    df.to_csv(output_path, index=False)
    return output_path


def compute_aggregated_scores(df, component_name):
    field_mappings = {
        'models': {
            'flagging': ['Article Flagging'], # 'screening', 
            'count': ['model_count', 'Model Counts', 'Model Count']
        },
        'outbreaks': {
            'flagging': ['Article Flagging'], # 'screening', 
            'count': ['outbreak_count', 'Outbreak Counts', 'Outbreak Count']
        },
        'parameters': {
            'flagging': ['Article Flagging'], # 'screening', 
            'count': ['parameter_count', 'Parameter Counts', 'Parameter Count']
        }
    }

    mapping = field_mappings.get(component_name, field_mappings['models'])
    flagging_fields = mapping['flagging']
    count_fields = mapping['count']

    records = []

    for model_name in df['model'].unique():
        model_df = df[df['model'] == model_name]

        for pathogen in model_df['pathogen'].unique():
            p_df = model_df[model_df['pathogen'] == pathogen]

            for metric_name, metric_col in [('F1', 'f1'), ('Precision', 'precision'), ('Recall', 'recall')]:
                flagging_data = p_df[p_df['field'].isin(flagging_fields)][metric_col]
                flagging_score = flagging_data.mean() if len(flagging_data) > 0 else np.nan

                count_data = p_df[p_df['field'].isin(count_fields)][metric_col]
                count_score = count_data.mean() if len(count_data) > 0 else np.nan

                excluded = flagging_fields + count_fields + ['Overall', 'overall']
                extraction_df = p_df[~p_df['field'].isin(excluded)]
                extraction_score = extraction_df[metric_col].mean() if len(extraction_df) > 0 else np.nan

                overall_score = np.nanmean([flagging_score, count_score, extraction_score])

                records.append({
                    'model': model_name,
                    'pathogen': pathogen,
                    'metric': metric_name,
                    'flagging_score': round(flagging_score, 3) if not np.isnan(flagging_score) else np.nan,
                    'count_score': round(count_score, 3) if not np.isnan(count_score) else np.nan,
                    'extraction_score': round(extraction_score, 3) if not np.isnan(extraction_score) else np.nan,
                    'overall_score': round(overall_score, 3) if not np.isnan(overall_score) else np.nan,
                })

    return pd.DataFrame(records)


def compute_model_summary(df, component_name):
    agg_df = compute_aggregated_scores(df, component_name)

    records = []
    for model_name in agg_df['model'].unique():
        model_df = agg_df[agg_df['model'] == model_name]

        for metric in ['F1', 'Precision', 'Recall']:
            metric_df = model_df[model_df['metric'] == metric]

            for category, col in [
                ('flagging', 'flagging_score'),
                ('count', 'count_score'),
                ('extraction', 'extraction_score'),
                ('overall', 'overall_score')
            ]:
                scores = metric_df[col].dropna().values
                if len(scores) > 0:
                    records.append({
                        'model': model_name,
                        'category': category,
                        'metric': metric,
                        'mean_score': round(np.mean(scores), 3),
                        'std_score': round(np.std(scores), 3),
                        'n_pathogens': len(scores)
                    })

    return pd.DataFrame(records)


def save_aggregated_metrics_csv(df, component_name, output_dir=None):
    output_dir = Path(output_dir) if output_dir else EVALS_BASE_DIR / component_name
    output_dir.mkdir(parents=True, exist_ok=True)

    agg_df = compute_aggregated_scores(df, component_name)
    aggregated_path = output_dir / f"extraction_{component_name}_aggregated.csv"
    agg_df.to_csv(aggregated_path, index=False)

    summary_df = compute_model_summary(df, component_name)
    summary_path = output_dir / f"extraction_{component_name}_model_summary.csv"
    summary_df.to_csv(summary_path, index=False)

    return aggregated_path, summary_path


def save_all_extraction_metrics(df, component_name, output_dir=None):
    output_dir = Path(output_dir) if output_dir else EVALS_BASE_DIR / component_name

    detailed_path = save_detailed_metrics_csv(df, component_name, output_dir)
    agg_path, summary_path = save_aggregated_metrics_csv(df, component_name, output_dir)

    return {
        'detailed': detailed_path,
        'aggregated': agg_path,
        'summary': summary_path
    }


def load_extraction_metrics(component_name, output_dir=None):
    output_dir = Path(output_dir) if output_dir else EVALS_BASE_DIR / component_name

    result = {}

    detailed_path = output_dir / f"extraction_{component_name}_detailed.csv"
    if detailed_path.exists():
        result['detailed'] = pd.read_csv(detailed_path)

    aggregated_path = output_dir / f"extraction_{component_name}_aggregated.csv"
    if aggregated_path.exists():
        result['aggregated'] = pd.read_csv(aggregated_path)

    summary_path = output_dir / f"extraction_{component_name}_model_summary.csv"
    if summary_path.exists():
        result['summary'] = pd.read_csv(summary_path)

    return result

---

# Model Extraction Evaluation

## Model Schema Definition

For model extraction evaluation, we define the following key fields:

$$\mathcal{F}_{\text{models}} = \{\texttt{model\_type}, \texttt{compartmental\_type}, \texttt{stoch\_deter}, \texttt{theoretical\_model}, \texttt{assumptions}, \texttt{interventions\_type}, \texttt{transmission\_route}\}$$

Each field has predefined valid values based on the PERG REDCap survey schema:
- **model_type**: Compartmental, Branching process, Agent/Individual based, Other, Unspecified
- **compartmental_type**: SIS, SIR, SEIR, SEIR-SEI, SAIR-SEI, Not compartmental, Other compartmental
- **stoch_deter**: Stochastic, Deterministic, Unspecified
- **transmission_route**: Multi-value field with routes such as Airborne, Human-to-human, Vector/Animal
- **assumptions**: Multi-value field with modelling assumptions
- **interventions_type**: Multi-value field with intervention categories

## Model Data Preparation

In [3]:
MODEL_TYPES = ['Compartmental', 'Branching process', 'Agent / Individual based', 'Other', 'Unspecified']

COMPARTMENTAL_TYPES = ['SIS', 'SIR', 'SEIR', 'SEIR-SEI', 'SAIR-SEI', 'Not compartmental', 'Other compartmental']

STOCH_DETER = ['Stochastic', 'Deterministic', 'Unspecified']

TRANSMISSION_ROUTES = ['Airborne or close contact', 'Human to human (direct contact)', 'Human to human (direct non-sexual contact)', 'Vector/Animal to human', 'Sexual', 'Unspecified']

ASSUMPTIONS = ['Homogeneous mixing', 'Latent period is same as incubation period', 'Heterogenity in transmission rates - between human groups', 'Heterogenity in transmission rates - between groups', 'Heterogenity in transmission rates - between human and vector', 'Heterogenity in transmission rates - over time', 'Age dependent susceptibility', 'Cross-immunity between Zika and dengue', 'Other', 'Unspecified']

INTERVENTIONS = ['Vaccination', 'Quarantine', 'Vector/Animal control', 'Treatment', 'Contact tracing', 'Hospitals', 'Treatment centres', 'Safe burials', 'Behaviour changes', 'Wolbachia replacement', 'Wolbachia suppression', 'Genetically modified mosquitoes', 'Mechanical removal of breeding sites', 'Pesticides/larvicides', 'Insecticide-treated nets', 'Indoor residual spraying', 'Other', 'Unspecified']

CODING_LANGUAGES = ['R', 'Python', 'Matlab', 'Julia', 'C++', 'Other']

DATA_AVAILABILITY = ['Yes - as an attachment', 'Yes - with a DOI', 'Yes - on Github', 'Yes - on another platform', 'Not available', 'Unspecified']

In [4]:
pathogens = ["Ebola", "Lassa", "SARS", "Zika"]
models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']

In [5]:
models_agentslr = {
    (pathogen, model_name): f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_models.csv' for pathogen in pathogens for model_name in models
}

fulltext_screening_agentslr = {
    (pathogen, model_name): f"../data/agentslr/client/{model_name}/{pathogen.lower()}/screening/fulltext_screening_results.csv"
    for pathogen in pathogens for model_name in models
}

perg_screening_files = {
    pathogen: f"../data/perg/screening/{pathogen}_filtered.csv"
    for pathogen in pathogens
}

perg_models_files = {
    pathogen: f"../data/perg/extracted/{pathogen.lower()}_models.csv"
    for pathogen in pathogens
}

### Article ID Mapping

In [ ]:
pathogens = ["Ebola", "Lassa", 'SARS', "Zika"]
models = ['oss','deepseek', 'kimi', 'gpt', 'glm']

for model_name in models:
    print(f"\n\nMODEL: {model_name}")
    for pathogen in pathogens:
        print(f"PATHOGEN: {pathogen}")

        df_agentslr_fulltext_screening = pd.read_csv(fulltext_screening_agentslr[(pathogen, model_name)])
        df_perg = pd.read_csv(perg_screening_files[pathogen])

        cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']

        for col in cols_to_be_added:
            if col in df_agentslr_fulltext_screening.columns:
                df_agentslr_fulltext_screening = df_agentslr_fulltext_screening.drop(columns=col)   

        df_agentslr_fulltext_screening = add_perg_columns_from_screening(df_agentslr_fulltext_screening, df_perg)

        df_agentslr_fulltext_screening['covidence_id'] = df_agentslr_fulltext_screening['Covidence #'].apply(lambda x: x.replace('#','').strip() if pd.notnull(x) else x)

        print(f"\nNumber of articles conditioned on PERG for {pathogen} and model {model_name}:", len(df_agentslr_fulltext_screening[df_agentslr_fulltext_screening.perg_fulltext_result=='INCLUDE']))

        df_agentslr_models = pd.read_csv(models_agentslr[(pathogen, model_name)])

        if 'article_id' in df_agentslr_models.columns and 'article_uuid' in df_agentslr_models.columns:
            print(f"Both article_id and article_uuid are present in df_agentslr_models for {pathogen} and model {model_name}. Skipping the merging process.")
            continue

        else:
            df_agentslr_models = df_agentslr_models[[col for col in df_agentslr_models.columns if 'covidence' not in col.lower()]]

            df_merged = df_agentslr_models.merge(
                df_agentslr_fulltext_screening[['article_id', 'covidence_id']],
                on='article_id',
                how='left'
            )

            df_merged = df_merged.rename(columns={'article_id': 'article_uuid'})
            df_merged = df_merged.rename(columns={'covidence_id': 'article_id'})

            print(f"Number of unique articles after merging with covidence_id:", len(df_merged.article_id.unique()))

            df_merged.to_csv(models_agentslr[(pathogen, model_name)], index=False)

## Model Evaluation Helper Functions

The following functions implement the evaluation framework for model extraction. Key components include:

1. **Field validators** - Define valid values for each schema field
2. **Similarity computation** - Calculate pairwise similarity between extractions using weighted field comparison
3. **Optimal matching** - Apply the Jonker-Volgenant algorithm to find optimal extraction pairs
4. **Metric computation** - Calculate precision, recall, and F1 for each evaluation dimension

In [6]:
FIELD_VALIDATORS = {
    'model_type': (MODEL_TYPES, False),
    'compartmental_type': (COMPARTMENTAL_TYPES, False),
    'stoch_deter': (STOCH_DETER, False),
    'transmission_route': (TRANSMISSION_ROUTES, True),
    'assumptions': (ASSUMPTIONS, True),
    'interventions_type': (INTERVENTIONS, True),
    'coding_language': (CODING_LANGUAGES, False),
    'is_data_used_available': (DATA_AVAILABILITY, False),
    'theoretical_model': (['True', 'False', 'true', 'false', '1', '0'], False),
    'code_available': (['True', 'False', 'true', 'false', '1', '0'], False),
    'spatial_model': (['True', 'False', 'true', 'false', '1', '0'], False),
    'spillover_included': (['True', 'False', 'true', 'false', '1', '0'], False),
    'uncertainty_was_considered': (['True', 'False', 'true', 'false', '1', '0'], False)
}


HIGH_WEIGHT_FIELDS = {
    'model_type': 1,
    'compartmental_type': 1,
    'stoch_deter': 1,
    'theoretical_model': 1,
    'assumptions': 1,
    'interventions_type': 1,
    'transmission_route': 1
}

FIELD_DISPLAY_NAMES = {
    'screening': 'Screening',
    'model_count': 'Model Counts',
    'model_type': 'Model Type',
    'compartmental_type': 'Compartmental Type',
    'stoch_deter': 'Stochastic vs Deterministic',
    'transmission_route': 'Transmission Routes',
    'theoretical_model': 'Theoretical vs Data-Fitted',
    'code_available': 'Code Available',
    'spatial_model': 'Spatial Model',
    'spillover_included': 'Spillover Included',
    'uncertainty_was_considered': 'Uncertainty Considered',
    'assumptions': 'Assumptions',
    'interventions_type': 'Interventions'
}

def normalise_weights(weights):
    total = sum(weights.values())
    return {k: v/total for k, v in weights.items()}

def is_valid_field_value(val, valid_values, is_multivalue):
    if pd.isna(val):
        return True
    
    valid_set = set(valid_values)
    
    if is_multivalue:
        items = [item.strip() for item in str(val).split(';') if item.strip()]
        return all(item in valid_set for item in items)
    else:
        return str(val).strip() in valid_set

def filter_invalid_rows(df, pathogen_name, filter_invalid=True):
    total_rows = len(df)
    
    if not filter_invalid:
        return df, 0, 0.0
    
    invalid_mask = pd.Series([False] * len(df), index=df.index)
    
    for field_name, (valid_values, is_multivalue) in FIELD_VALIDATORS.items():
        if field_name not in df.columns:
            continue
        
        for idx, row in df.iterrows():
            val = row[field_name]
            if not is_valid_field_value(val, valid_values, is_multivalue):
                invalid_mask[idx] = True
    
    invalid_count = invalid_mask.sum()
    invalid_pct = (invalid_count / total_rows * 100) if total_rows > 0 else 0
    
    df_filtered = df[~invalid_mask].copy()
    
    return df_filtered, invalid_count, invalid_pct

def load_data(pathogen, filter_invalid=True, perg_extractions_path=None, extracted_extractions_path=None):

    if perg_extractions_path is None or extracted_extractions_path is None:
        raise ValueError("Both perg_extractions_path and extracted_extractions_path must be provided.")

    perg = pd.read_csv(perg_extractions_path)
    extracted = pd.read_csv(extracted_extractions_path)
    
    perg['article_id'] = perg['covidence_id']
    
    perg_filtered, perg_invalid_count, perg_invalid_pct = filter_invalid_rows(perg, pathogen, filter_invalid)
    extracted_filtered, extracted_invalid_count, extracted_invalid_pct = filter_invalid_rows(extracted, pathogen, filter_invalid)
    
    common_articles = set(perg_filtered['article_id'].unique()) & set(extracted_filtered['article_id'].unique())
    
    perg_filtered = perg_filtered[perg_filtered['article_id'].isin(common_articles)]
    extracted_filtered = extracted_filtered[extracted_filtered['article_id'].isin(common_articles)]
    
    filter_stats = {
        'pathogen': pathogen,
        'perg_total': len(perg),
        'perg_invalid': perg_invalid_count,
        'perg_invalid_pct': round(perg_invalid_pct, 2),
        'extracted_total': len(extracted),
        'extracted_invalid': extracted_invalid_count,
        'extracted_invalid_pct': round(extracted_invalid_pct, 2)
    }
    
    return perg_filtered, extracted_filtered, list(common_articles), filter_stats

def compute_field_similarity(val1, val2, field_name, is_multivalue=False):
    if pd.isna(val1) and pd.isna(val2):
        return 1.0
    if pd.isna(val1) or pd.isna(val2):
        return 0.0
    
    if is_multivalue:
        set1 = set(str(val1).split(';'))
        set2 = set(str(val2).split(';'))
        if len(set1) == 0 and len(set2) == 0:
            return 1.0
        if len(set1) == 0 or len(set2) == 0:
            return 0.0
        intersection = len(set1 & set2)
        union = len(set1 | set2)
        return intersection / union if union > 0 else 0.0
    else:
        return 1.0 if str(val1).strip() == str(val2).strip() else 0.0

def compute_model_similarity(perg_row, extracted_row):
    weights = normalise_weights(HIGH_WEIGHT_FIELDS)
    
    multivalue_fields = {'transmission_route', 'assumptions', 'interventions_type'}
    
    similarity = 0.0
    for field, weight in weights.items():
        if field not in perg_row.index or field not in extracted_row.index:
            continue
        is_multivalue = field in multivalue_fields
        field_sim = compute_field_similarity(perg_row[field], extracted_row[field], field, is_multivalue)
        similarity += weight * field_sim
    
    return similarity

def match_models_in_article(perg_models, extracted_models):
    n_perg = len(perg_models)
    n_extracted = len(extracted_models)
    
    if n_perg == 0 or n_extracted == 0:
        return []
    
    similarity_matrix = np.zeros((n_perg, n_extracted))
    
    for i, (_, perg_row) in enumerate(perg_models.iterrows()):
        for j, (_, extracted_row) in enumerate(extracted_models.iterrows()):
            similarity_matrix[i, j] = compute_model_similarity(perg_row, extracted_row)
    
    cost_matrix = 1 - similarity_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    matches = []
    for i, j in zip(row_ind, col_ind):
        matches.append({
            'perg_idx': perg_models.index[i],
            'extracted_idx': extracted_models.index[j],
            'similarity': similarity_matrix[i, j]
        })
    
    return matches

def evaluate_screening_optimal(perg, extracted, common_articles):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_count = len(perg[perg['article_id'] == article_id])
        extracted_count = len(extracted[extracted['article_id'] == article_id])
        
        tp = min(perg_count, extracted_count)
        fp = max(0, extracted_count - perg_count)
        fn = max(0, perg_count - extracted_count)
        
        tp_total += tp
        fp_total += fp
        fn_total += fn
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

def evaluate_field_optimal(perg, extracted, common_articles, field_name, is_multivalue):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_models = perg[perg['article_id'] == article_id]
        extracted_models = extracted[extracted['article_id'] == article_id]
        
        if field_name == 'compartmental_type':
            perg_models = perg_models[perg_models['model_type'] == 'Compartmental']
            extracted_models = extracted_models[extracted_models['model_type'] == 'Compartmental']
        
        matches = match_models_in_article(perg_models, extracted_models)
        
        for match in matches:
            perg_val = perg.loc[match['perg_idx'], field_name]
            extracted_val = extracted.loc[match['extracted_idx'], field_name]
            
            if is_multivalue:
                perg_set = (
                    {t.strip() for t in str(perg_val).split(';') if t.strip()}
                    if not pd.isna(perg_val)
                    else set()
                )
                extracted_set = (
                    {t.strip() for t in str(extracted_val).split(';') if t.strip()}
                    if not pd.isna(extracted_val)
                    else set()
                )
                
                tp = len(perg_set & extracted_set)
                fp = len(extracted_set - perg_set)
                fn = len(perg_set - extracted_set)
                
                tp_total += tp
                fp_total += fp
                fn_total += fn
            else:
                if pd.isna(perg_val) and pd.isna(extracted_val):
                    tp_total += 1
                elif pd.isna(perg_val) or pd.isna(extracted_val):
                    if not pd.isna(extracted_val):
                        fp_total += 1
                    if not pd.isna(perg_val):
                        fn_total += 1
                elif str(perg_val).strip() == str(extracted_val).strip():
                    tp_total += 1
                else:
                    fp_total += 1
                    fn_total += 1
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

def get_article_level_sets(df, field_name, is_multivalue=False):
    if len(df) == 0:
        return set()
    
    values = set()
    
    for _, row in df.iterrows():
        val = row.get(field_name)
        
        if pd.isna(val):
            continue
        
        if is_multivalue:
            items = str(val).split(';')
            values.update([item.strip() for item in items if item.strip()])
        else:
            values.add(str(val).strip())
    
    return values

def evaluate_screening_sweep(perg, extracted, common_articles):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_count = len(perg[perg['article_id'] == article_id])
        extracted_count = len(extracted[extracted['article_id'] == article_id])
        
        tp = min(perg_count, extracted_count)
        fp = max(0, extracted_count - perg_count)
        fn = max(0, perg_count - extracted_count)
        
        tp_total += tp
        fp_total += fp
        fn_total += fn
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

def evaluate_field_sweep(perg, extracted, common_articles, field_name, is_multivalue):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_models = perg[perg['article_id'] == article_id]
        extracted_models = extracted[extracted['article_id'] == article_id]
        
        if field_name == 'compartmental_type':
            perg_models = perg_models[perg_models['model_type'] == 'Compartmental']
            extracted_models = extracted_models[extracted_models['model_type'] == 'Compartmental']
        
        perg_set = get_article_level_sets(perg_models, field_name, is_multivalue)
        extracted_set = get_article_level_sets(extracted_models, field_name, is_multivalue)
        
        tp = len(perg_set & extracted_set)
        fp = len(extracted_set - perg_set)
        fn = len(perg_set - extracted_set)
        
        tp_total += tp
        fp_total += fp
        fn_total += fn
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

def evaluate_pathogen(pathogen, method='sweep', filter_invalid=True, perg_extractions_path=None, extracted_extractions_path=None):
    perg, extracted, common_articles, filter_stats = load_data(pathogen, filter_invalid, perg_extractions_path, extracted_extractions_path)
    
    results = []
    
    if method == 'optimal':
        screening_metrics = evaluate_screening_optimal(perg, extracted, common_articles)
        results.append({
            'pathogen': pathogen,
            'field': 'screening',
            'precision': screening_metrics['precision'],
            'recall': screening_metrics['recall'],
            'f1': screening_metrics['f1']
        })
        
        model_count_metrics = evaluate_screening_optimal(perg, extracted, common_articles)
        results.append({
            'pathogen': pathogen,
            'field': 'model_count',
            'precision': model_count_metrics['precision'],
            'recall': model_count_metrics['recall'],
            'f1': model_count_metrics['f1']
        })
        
        fields_to_evaluate = [
            ('model_type', False),
            ('compartmental_type', False),
            ('stoch_deter', False),
            ('transmission_route', True),
            ('theoretical_model', False),
            ('code_available', False),
            ('spatial_model', False),
            ('spillover_included', False),
            ('uncertainty_was_considered', False),
            ('assumptions', True),
            ('interventions_type', True)
        ]
        
        for field_name, is_multivalue in fields_to_evaluate:
            if field_name not in perg.columns or field_name not in extracted.columns:
                continue

            metrics = evaluate_field_optimal(perg, extracted, common_articles, field_name, is_multivalue)

            results.append({
                'pathogen': pathogen,
                'field': field_name,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1']
            })
    
    else:
        screening_metrics = evaluate_screening_sweep(perg, extracted, common_articles)
        results.append({
            'pathogen': pathogen,
            'field': 'screening',
            'precision': screening_metrics['precision'],
            'recall': screening_metrics['recall'],
            'f1': screening_metrics['f1']
        })
        
        model_count_metrics = evaluate_screening_sweep(perg, extracted, common_articles)
        results.append({
            'pathogen': pathogen,
            'field': 'model_count',
            'precision': model_count_metrics['precision'],
            'recall': model_count_metrics['recall'],
            'f1': model_count_metrics['f1']
        })
        
        fields_to_evaluate = [
            ('model_type', False),
            ('compartmental_type', False),
            ('stoch_deter', False),
            ('transmission_route', True),
            ('theoretical_model', False),
            ('code_available', False),
            ('spatial_model', False),
            ('spillover_included', False),
            ('uncertainty_was_considered', False),
            ('assumptions', True),
            ('interventions_type', True)
        ]
        
        for field_name, is_multivalue in fields_to_evaluate:
            if field_name not in perg.columns or field_name not in extracted.columns:
                continue
            
            metrics = evaluate_field_sweep(perg, extracted, common_articles, field_name, is_multivalue)
            
            results.append({
                'pathogen': pathogen,
                'field': field_name,
                'precision': metrics['precision'],
                'recall': metrics['recall'],
                'f1': metrics['f1']
            })
    
    return results, filter_stats

def create_summary_table(results_df):
    all_fields = results_df['field'].unique()
    pathogens = sorted(results_df['pathogen'].unique())

    summary_data = []

    for field in all_fields:
        if field == 'screening':
            continue

        row = {'Field': FIELD_DISPLAY_NAMES.get(field, field)}

        for pathogen in pathogens:
            field_data = results_df[(results_df['pathogen'] == pathogen) & (results_df['field'] == field)]
            if len(field_data) > 0:
                p = field_data.iloc[0]['precision']
                r = field_data.iloc[0]['recall']
                row[f"{pathogen}_P"] = p
                row[f"{pathogen}_R"] = r
                row[f"{pathogen}_F1"] = round(_f1_from_pr(p, r), 3)
            else:
                row[f"{pathogen}_P"] = np.nan
                row[f"{pathogen}_R"] = np.nan
                row[f"{pathogen}_F1"] = np.nan

        summary_data.append(row)

    summary = pd.DataFrame(summary_data)

    field_order = [
        'Model Counts', 'Model Type', 'Compartmental Type',
        'Stochastic vs Deterministic', 'Theoretical vs Data-Fitted',
        'Code Available', 'Spatial Model', 'Spillover Included',
        'Uncertainty Considered', 'Transmission Routes', 'Assumptions', 'Interventions'
    ]

    summary['Field'] = pd.Categorical(summary['Field'], categories=field_order, ordered=True)
    summary = summary.sort_values('Field').reset_index(drop=True)

    overall_row = {'Field': 'Overall'}
    for pathogen in pathogens:
        pathogen_data = results_df[results_df['pathogen'] == pathogen]
        extraction_data = pathogen_data[~pathogen_data['field'].isin(['screening', 'model_count'])]

        p_mean = extraction_data['precision'].mean()
        r_mean = extraction_data['recall'].mean()

        overall_row[f"{pathogen}_P"] = round(p_mean, 3)
        overall_row[f"{pathogen}_R"] = round(r_mean, 3)
        overall_row[f"{pathogen}_F1"] = round(_f1_from_pr(p_mean, r_mean), 3)

    summary = pd.concat([summary, pd.DataFrame([overall_row])], ignore_index=True)
    return summary

## Model Extraction Results

In [7]:
METHOD = 'optimal'
FILTER_INVALID = True

all_results = []
all_filter_stats = []

for pathogen in ['Ebola','Lassa', 'SARS', 'Zika']:
    print(f"Evaluating {pathogen}...")
    perg = f'../data/perg/extracted/{pathogen.lower()}_models.csv'
    extracted = f'../data/agentslr/client/oss/{pathogen.lower()}/extractions/data_extraction_models.csv'

    pathogen_results, filter_stats = evaluate_pathogen(pathogen, method=METHOD, filter_invalid=FILTER_INVALID,perg_extractions_path=perg, extracted_extractions_path=extracted)
    
    all_results.extend(pathogen_results)
    all_filter_stats.append(filter_stats)

results_df = pd.DataFrame(all_results)
filter_stats_df = pd.DataFrame(all_filter_stats)

if FILTER_INVALID:
    print("\n" + "="*100)
    print("INVALID ROWS FILTERING STATISTICS")
    print("="*100)
    print(filter_stats_df.to_string(index=False))

summary_table = create_summary_table(results_df)

Evaluating Ebola...
Evaluating Lassa...
Evaluating SARS...
Evaluating Zika...

INVALID ROWS FILTERING STATISTICS
pathogen  perg_total  perg_invalid  perg_invalid_pct  extracted_total  extracted_invalid  extracted_invalid_pct
   Ebola         294            46             15.65              239                  0                    0.0
   Lassa          52             2              3.85               19                  0                    0.0
    SARS         112             8              7.14               85                  0                    0.0
    Zika         229            53             23.14              132                  0                    0.0


In [20]:
def compute_model_flagging_metrics(
    pathogen,
    model_name,
    fulltext_screening_path,
    perg_screening_path,
    models_extraction_path,
    perg_models_files
):
    perg_screening_path = perg_screening_files[pathogen]
    fulltext_screening_path = fulltext_screening_agentslr[(pathogen, model_name)]
    models_extraction_path = models_agentslr[(pathogen, model_name)]
    perg_models_path = perg_models_files[pathogen]

    df_fulltext = pd.read_csv(fulltext_screening_path)
    df_perg_screening = pd.read_csv(perg_screening_path)
    df_models = pd.read_csv(models_extraction_path)
    df_perg_models = pd.read_csv(perg_models_path)

    cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']
    for col in cols_to_be_added:
        if col in df_fulltext.columns:
            df_fulltext = df_fulltext.drop(columns=col)

    df_fulltext = add_perg_columns_from_screening(df_fulltext, df_perg_screening)

    df_relevant = df_fulltext[df_fulltext['perg_fulltext_result'] == 'INCLUDE'].copy()

    df_relevant['covidence_id'] = df_relevant['Covidence #'].apply(
        lambda x: x.replace('#', '').strip() if pd.notnull(x) else x
    )

    flagged_positive_cov_id = set(map(int, df_models['article_id'].dropna().unique()))

    true_all_perg = set(map(int, df_perg_models['covidence_id'].dropna().unique()))

    df_relevant['screening_ai_flag'] = df_relevant['covidence_id'].apply(
        lambda v: True if pd.notnull(v) and int(str(v).replace('#', '').strip()) in flagged_positive_cov_id else False
    )

    df_relevant['true_perg_flag'] = df_relevant['covidence_id'].apply(
        lambda v: True if pd.notnull(v) and int(str(v).replace('#', '').strip()) in true_all_perg else False
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        df_relevant["true_perg_flag"],
        df_relevant["screening_ai_flag"],
        average='macro',
    )

    return {
            'pathogen': pathogen,
            'model': model_name,
            'field': 'Article Flagging',
            'precision': round(precision, 3),
            'recall': round(recall, 3),
            'f1': round(f1, 3),
            # 'n_articles': len(df_relevant)
        }

In [16]:
model = 'oss'

flagging_metrics_results = []

for pathogen in ['Ebola', 'Lassa', 'SARS', 'Zika']:
    metrics = compute_model_flagging_metrics(
        pathogen=pathogen,
        model_name=model,
        fulltext_screening_path=fulltext_screening_agentslr,
        perg_screening_path=perg_screening_files,
        models_extraction_path=models_agentslr,
        perg_models_files=perg_models_files
    )
    flagging_metrics_results.append(metrics)

In [17]:
pd.DataFrame(flagging_metrics_results)

,pathogen,model,field,precision,recall,f1,n_articles
0,Ebola,oss,Article Flagging,0.915,0.917,0.915,223
1,Lassa,oss,Article Flagging,0.955,0.987,0.970,49
2,SARS,oss,Article Flagging,0.861,0.861,0.861,103
3,Zika,oss,Article Flagging,0.868,0.892,0.876,166


## Model Extraction Ablation Study

In [28]:
from tqdm import tqdm

models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']
df_all = []
METHOD = 'optimal'
FILTER_INVALID = True

for model_name in tqdm(models, desc="Evaluating models"):
    all_results = []
    
    for pathogen in ['Ebola', 'Lassa', 'SARS', 'Zika']:
        perg = f'../data/perg/extracted/{pathogen.lower()}_models.csv'
        extracted = f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_models.csv'
        
        pathogen_results, _ = evaluate_pathogen(
            pathogen, method=METHOD, filter_invalid=FILTER_INVALID,
            perg_extractions_path=perg, extracted_extractions_path=extracted
        )

        for result in pathogen_results:
            result['model'] = model_name

        all_results.extend(pathogen_results)

        flagging_metrics = compute_model_flagging_metrics(
                    pathogen=pathogen,
                    model_name=model_name,
                    fulltext_screening_path=fulltext_screening_agentslr,
                    perg_screening_path=perg_screening_files,
                    models_extraction_path=models_agentslr,
                    perg_models_files=perg_models_files
                )
        
        all_results.append(flagging_metrics)
    
    df_all.append(pd.DataFrame(all_results))

df_model_extractions_all = pd.concat(df_all, ignore_index=True)

Evaluating models:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluating models: 100%|██████████| 5/5 [01:24<00:00, 16.83s/it]


In [29]:
df_model_extractions_all.sample(10)

,pathogen,field,precision,recall,f1,model
116,SARS,interventions_type,0.263,0.656,0.375,kimi
20,Lassa,Article Flagging,0.955,0.987,0.970,oss
39,Zika,code_available,0.821,0.762,0.790,oss
59,Lassa,theoretical_model,0.500,0.500,0.500,deepseek
45,Ebola,model_type,0.757,0.757,0.757,deepseek
146,Lassa,code_available,0.875,0.778,0.824,glm
190,Lassa,assumptions,0.455,0.909,0.606,gpt
121,Zika,compartmental_type,0.760,0.760,0.760,kimi
49,Ebola,code_available,0.814,0.814,0.814,deepseek
168,Zika,code_available,0.824,0.757,0.789,glm


In [35]:
df_model_extractions_all[df_model_extractions_all.model=='deepseek']

,pathogen,field,precision,recall,f1,model
43,Ebola,screening,0.805,0.986,0.886,deepseek
44,Ebola,model_count,0.805,0.986,0.886,deepseek
45,Ebola,model_type,0.757,0.757,0.757,deepseek
46,Ebola,stoch_deter,0.786,0.948,0.859,deepseek
47,Ebola,transmission_route,0.113,0.132,0.122,deepseek
48,Ebola,theoretical_model,0.743,0.743,0.743,deepseek
49,Ebola,code_available,0.814,0.814,0.814,deepseek
50,Ebola,assumptions,0.235,0.395,0.295,deepseek
51,Ebola,interventions_type,0.337,0.447,0.384,deepseek
52,Ebola,Article Flagging,0.866,0.864,0.864,deepseek


In [31]:
df_model_extractions_all.to_csv('../data/agentslr/evals/data_extraction/models/all_model_extractions_results.csv', index=False)

In [32]:
records = []
for model_name, model_df in df_model_extractions_all[(df_model_extractions_all.field != 'screening') & (df_model_extractions_all.field != 'Article Flagging')].groupby('model'):
    avg_precision = model_df['precision'].mean()
    avg_recall = model_df['recall'].mean()
    avg_f1 = model_df['f1'].mean()
    
    records.append({
        'model': model_name,
        'avg_precision': round(avg_precision, 3),
        'avg_recall': round(avg_recall, 3),
        'avg_f1': round(avg_f1, 3)
    })

In [33]:
pd.DataFrame(records)

,model,avg_precision,avg_recall,avg_f1
0,deepseek,0.629,0.679,0.649
1,glm,0.659,0.728,0.682
2,gpt,0.644,0.757,0.675
3,kimi,0.650,0.751,0.686
4,oss,0.633,0.737,0.669


## Save Model Extraction CSVs

In [34]:
save_all_extraction_metrics(df_model_extractions_all, 'models', EVALS_MODELS_DIR)
print("Model extraction metrics saved!")

Model extraction metrics saved!


---

# Outbreak Extraction Evaluation

## Outbreak Schema Definition

For outbreak extraction evaluation, we define key fields based on their discriminative power for identifying unique outbreak events:

$$\mathcal{F}_{\text{outbreaks}} = \{\texttt{outbreak\_start\_day}, \texttt{outbreak\_start\_month}, \texttt{outbreak\_start\_year}, \texttt{outbreak\_end\_day}, \texttt{outbreak\_end\_month}, \texttt{outbreak\_end\_year}, \texttt{cases\_confirmed}, \texttt{deaths}, \texttt{outbreak\_country}, \texttt{outbreak\_location}, \texttt{detection\_mode}, \texttt{pre\_outbreak\_status}\}$$

Weights are determined by the discriminative power of each field for identifying unique outbreak events:
- **High weight (1.0)**: outbreak_country, outbreak_start_year, cases_confirmed, deaths
- **Medium weight (0.6-0.8)**: outbreak_start_month, outbreak_end_year
- **Lower weight (0.5-0.7)**: outbreak_location, mode_of_detection

Fields are grouped into four categories:
- **Temporal Features** (7 fields): outbreak start/end dates and duration
- **Geographic and Spatial Features** (2 fields): country and specific location
- **Case Burden** (5 fields): confirmed, suspected, asymptomatic, severe cases, and deaths
- **Epidemiological Context** (3 fields): detection mode, pre-outbreak status, asymptomatic transmission

## Outbreak Data Preparation

In [20]:
pathogens = ["Lassa", "Zika"]
models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']

In [21]:
outbreaks_agentslr = {
    (pathogen, model_name): f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_outbreaks.csv' for pathogen in pathogens for model_name in models
}

fulltext_screening_agentslr = {
    (pathogen, model_name): f"../data/agentslr/client/{model_name}/{pathogen.lower()}/screening/fulltext_screening_results.csv"
    for pathogen in pathogens for model_name in models
}

perg_screening_files = {
    pathogen: f"../data/perg/screening/{pathogen}_filtered.csv"
    for pathogen in pathogens
}

perg_outbreaks_files = {
    pathogen: f"../data/perg/extracted/{pathogen.lower()}_outbreaks.csv"
    for pathogen in pathogens
}

### Outbreak Article ID Mapping

In [18]:
pathogens = ["Lassa", "Zika"]
models = ['oss','deepseek', 'kimi', 'gpt', 'glm']

for model_name in models:
    print(f"\n\nMODEL: {model_name}")
    for pathogen in pathogens:
        print(f"PATHOGEN: {pathogen}")

        df_agentslr_fulltext_screening = pd.read_csv(fulltext_screening_agentslr[(pathogen, model_name)])
        df_perg = pd.read_csv(perg_screening_files[pathogen])

        cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']

        for col in cols_to_be_added:
            if col in df_agentslr_fulltext_screening.columns:
                df_agentslr_fulltext_screening = df_agentslr_fulltext_screening.drop(columns=col)   

        df_agentslr_fulltext_screening = add_perg_columns_from_screening(df_agentslr_fulltext_screening, df_perg)

        df_agentslr_fulltext_screening['covidence_id'] = df_agentslr_fulltext_screening['Covidence #'].apply(lambda x: x.replace('#','').strip() if pd.notnull(x) else x)

        print(f"\nNumber of articles conditioned on PERG for {pathogen} and model {model_name}:", len(df_agentslr_fulltext_screening[df_agentslr_fulltext_screening.perg_fulltext_result=='INCLUDE']))

        df_agentslr_outbreaks = pd.read_csv(outbreaks_agentslr[(pathogen, model_name)])

        if 'article_id' in df_agentslr_outbreaks.columns and 'article_uuid' in df_agentslr_outbreaks.columns:
            print(f"Both article_id and article_uuid are present in df_agentslr_outbreaks for {pathogen} and model {model_name}. Skipping the merging process.")
            continue

        else:
            df_agentslr_outbreaks = df_agentslr_outbreaks[[col for col in df_agentslr_outbreaks.columns if 'covidence' not in col.lower()]]

            df_merged = df_agentslr_outbreaks.merge(
                df_agentslr_fulltext_screening[['article_id', 'covidence_id']],
                on='article_id',
                how='left'
            )

            df_merged = df_merged.rename(columns={'article_id': 'article_uuid'})
            df_merged = df_merged.rename(columns={'covidence_id': 'article_id'})

            print(f"Number of unique articles after merging with covidence_id:", len(df_merged.article_id.unique()))

            df_merged.to_csv(outbreaks_agentslr[(pathogen, model_name)], index=False)



MODEL: oss
PATHOGEN: Lassa

Number of articles conditioned on PERG for Lassa and model oss: 49
Both article_id and article_uuid are present in df_agentslr_outbreaks for Lassa and model oss. Skipping the merging process.
PATHOGEN: Zika

Number of articles conditioned on PERG for Zika and model oss: 185
Both article_id and article_uuid are present in df_agentslr_outbreaks for Zika and model oss. Skipping the merging process.


MODEL: deepseek
PATHOGEN: Lassa

Number of articles conditioned on PERG for Lassa and model deepseek: 49
Both article_id and article_uuid are present in df_agentslr_outbreaks for Lassa and model deepseek. Skipping the merging process.
PATHOGEN: Zika

Number of articles conditioned on PERG for Zika and model deepseek: 185
Both article_id and article_uuid are present in df_agentslr_outbreaks for Zika and model deepseek. Skipping the merging process.


MODEL: kimi
PATHOGEN: Lassa

Number of articles conditioned on PERG for Lassa and model kimi: 49
Both article_id an

## Outbreak Evaluation Helper Functions

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

COUNTRIES = [
    'Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 
    'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 
    'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 
    'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 
    'Bolivia (Plurinational State of)', 'Bosnia and Herzegovina', 'Botswana', 
    'Brazil', 'Brunei Darussalam', 'Bulgaria', 'Burkina Faso', 'Burundi', 
    'Cabo Verde', 'Cambodia', 'Cameroon', 'Canada', 'Central African Republic', 
    'Chad', 'Chile', 'China', 'Colombia', 'Comoros', 'Congo', 'Cook Islands', 
    'Costa Rica', "Côte d'Ivoire", 'Croatia', 'Cuba', 'Cyprus', 
    'Czechia', "Democratic People's Republic of Korea", 
    'Democratic Republic of the Congo', 'Denmark', 'Djibouti', 'Dominica', 
    'Dominican Republic', 'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 
    'Eritrea', 'Estonia', 'Eswatini', 'Ethiopia', 'Fiji', 'Finland', 'France', 
    'Gabon', 'Gambia', 'Georgia', 'Germany', 'Ghana', 'Greece', 'Grenada', 
    'Guatemala', 'Guinea', 'Guinea-Bissau', 'Guyana', 'Haiti', 'Honduras', 
    'Hungary', 'Iceland', 'India', 'Indonesia', 'Iran (Islamic Republic of)', 
    'Iraq', 'Ireland', 'Israel', 'Italy', 'Jamaica', 'Japan', 'Jordan', 
    'Kazakhstan', 'Kenya', 'Kiribati', 'Kuwait', 'Kyrgyzstan', 
    "Lao People's Democratic Republic", 'Latvia', 'Lebanon', 'Lesotho', 
    'Liberia', 'Libya', 'Lithuania', 'Luxembourg', 'Madagascar', 'Malawi', 
    'Malaysia', 'Maldives', 'Mali', 'Malta', 'Marshall Islands', 'Mauritania', 
    'Mauritius', 'Mexico', 'Federated States of Micronesia', 'Monaco', 
    'Mongolia', 'Montenegro', 'Morocco', 'Mozambique', 'Myanmar', 'Namibia', 
    'Nauru', 'Nepal', 'Netherlands', 'New Zealand', 'Nicaragua', 'Niger', 
    'Nigeria', 'Niue', 'North Macedonia', 'Norway', 'Oman', 'Pakistan', 
    'Palau', 'Panama', 'Papua New Guinea', 'Paraguay', 'Peru', 'Philippines', 
    'Poland', 'Portugal', 'Qatar', 'Republic of Korea', 'Republic of Moldova', 
    'Romania', 'Russian Federation', 'Rwanda', 'Saint Kitts and Nevis', 
    'Saint Lucia', 'Saint Vincent and the Grenadines', 'Samoa', 'San Marino', 
    'Sao Tome and Principe', 'Saudi Arabia', 'Senegal', 'Serbia', 'Seychelles', 
    'Sierra Leone', 'Singapore', 'Slovakia', 'Slovenia', 'Solomon Islands', 
    'Somalia', 'South Africa', 'South Sudan', 'Spain', 'Sri Lanka', 'Sudan', 
    'Suriname', 'Sweden', 'Switzerland', 'Syrian Arab Republic', 'Tajikistan', 
    'Thailand', 'Timor-Leste', 'Togo', 'Tonga', 'Trinidad and Tobago', 
    'Tunisia', 'Türkiye', 'Turkmenistan', 'Tuvalu', 'Uganda', 'Ukraine', 
    'United Arab Emirates', 
    'United Kingdom of Great Britain and Northern Ireland', 
    'United Republic of Tanzania', 'United States of America', 'Uruguay', 
    'Uzbekistan', 'Vanuatu', 'Venezuela (Bolivarian Republic of)', 'Viet Nam', 
    'Yemen', 'Yugoslavia', 'Zambia', 'Zimbabwe'
]

MODE_OF_DETECTION = [
    "Molecular (PCR etc)",
    "Symptoms",
    "Confirmed + Suspected",
    "Unspecified"
]

PRE_OUTBREAK_STATUS = [
    "Disease-free baseline",
    "Endemic equilibrium",
    "Unspecified",
    "Probable"
]

FIELD_VALIDATORS = {
    'outbreak_country': (COUNTRIES, False),
    'outbreak_start_month': (MONTHS, False),
    'outbreak_end_month': (MONTHS, False),
    'mode_of_detection': (MODE_OF_DETECTION, False),
    'pre_outbreak': (PRE_OUTBREAK_STATUS, False),
    'outbreak_is_currently_ongoing': (['True', 'False', 'true', 'false', '1', '0', True, False], False),
    'asymptomatic_transmission_described': (['True', 'False', 'true', 'false', '1', '0', True, False], False),
}

OUTBREAK_HIGH_WEIGHT_FIELDS = {
    'outbreak_start_day': 0.5,
    'outbreak_start_month': 0.6,
    'outbreak_start_year': 1.0,
    'outbreak_end_day': 0.5,
    'outbreak_end_month': 0.6,
    'outbreak_end_year': 0.8,
    'cases_confirmed': 1.0,
    'deaths': 1.0,
    'outbreak_country': 1.0,
    'outbreak_location': 0.5,
    'mode_of_detection': 0.7,
    'pre_outbreak': 0.5,
    'outbreak_is_currently_ongoing': 0.4,
    'asymptomatic_transmission_described': 0.4,
}


In [24]:
def normalise_weights(weights: dict) -> dict:
    total = sum(weights.values())
    return {k: v / total for k, v in weights.items()} if total else dict(weights)

def is_valid_field_value(val, valid_values, is_multivalue: bool) -> bool:
    if pd.isna(val):
        return True
    valid_set = set(str(v) for v in valid_values)
    if is_multivalue:
        items = [item.strip() for item in str(val).split(';') if item.strip()]
        return all(item in valid_set for item in items)
    return str(val).strip() in valid_set

def filter_invalid_outbreak_rows(df: pd.DataFrame, filter_invalid: bool = True):
    total_rows = len(df)
    if not filter_invalid:
        return df, 0, 0.0

    invalid_mask = pd.Series(False, index=df.index)

    for field_name, (valid_values, is_multivalue) in FIELD_VALIDATORS.items():
        if field_name not in df.columns:
            continue
        for idx, row in df.iterrows():
            if not is_valid_field_value(row[field_name], valid_values, is_multivalue):
                invalid_mask[idx] = True

    invalid_count = int(invalid_mask.sum())
    invalid_pct = (invalid_count / total_rows * 100) if total_rows > 0 else 0.0
    return df.loc[~invalid_mask].copy(), invalid_count, invalid_pct

def normalize_boolean(val):
    if pd.isna(val):
        return None
    if val in [True, 'True', 'true', '1', 1]:
        return True
    if val in [False, 'False', 'false', '0', 0]:
        return False
    return None

def load_outbreak_data(pathogen: str, filter_invalid: bool = True, perg_extractions_path=None, extracted_extractions_path=None):
    if perg_extractions_path is None or extracted_extractions_path is None:
        raise ValueError("Both perg_extractions_path and extracted_extractions_path must be provided.")

    perg = pd.read_csv(perg_extractions_path)
    extracted = pd.read_csv(extracted_extractions_path)

    perg['article_id'] = perg['covidence_id']

    # PERG column aliases (OLD)
    if 'asymptomatic_transmission_described' not in perg.columns and 'asymptomatic_transmission' in perg.columns:
        perg['asymptomatic_transmission_described'] = perg['asymptomatic_transmission']
    if 'outbreak_is_currently_ongoing' not in perg.columns and 'ongoing' in perg.columns:
        perg['outbreak_is_currently_ongoing'] = perg['ongoing']
    if 'mode_of_detection' not in perg.columns and 'cases_mode_detection' in perg.columns:
        perg['mode_of_detection'] = perg['cases_mode_detection']
    if 'outbreak_start_year' not in perg.columns and 'outbreak_date_year' in perg.columns:
        perg['outbreak_start_year'] = perg['outbreak_date_year']

    perg_filtered, perg_invalid_count, perg_invalid_pct = filter_invalid_outbreak_rows(perg, filter_invalid)
    extracted_filtered, extracted_invalid_count, extracted_invalid_pct = filter_invalid_outbreak_rows(extracted, filter_invalid)

    common_articles = set(perg_filtered['article_id'].unique()) & set(extracted_filtered['article_id'].unique())

    perg_filtered = perg_filtered[perg_filtered['article_id'].isin(common_articles)]
    extracted_filtered = extracted_filtered[extracted_filtered['article_id'].isin(common_articles)]

    filter_stats = {
        'pathogen': pathogen,
        'perg_total': len(perg),
        'perg_invalid': perg_invalid_count,
        'perg_invalid_pct': round(perg_invalid_pct, 2),
        'extracted_total': len(extracted),
        'extracted_invalid': extracted_invalid_count,
        'extracted_invalid_pct': round(extracted_invalid_pct, 2),
    }

    return perg_filtered, extracted_filtered, list(common_articles), filter_stats

def compute_outbreak_field_similarity(val1, val2, field_name: str):
    if pd.isna(val1) and pd.isna(val2):
        return 1.0
    if pd.isna(val1) or pd.isna(val2):
        return 0.0

    if field_name in ['asymptomatic_transmission_described', 'outbreak_is_currently_ongoing']:
        b1 = normalize_boolean(val1)
        b2 = normalize_boolean(val2)
        if b1 is None or b2 is None:
            return 0.0
        return 1.0 if b1 == b2 else 0.0

    if field_name in [
        'outbreak_start_day', 'outbreak_end_day', 'outbreak_start_year', 'outbreak_end_year',
        'cases_confirmed', 'deaths'
    ]:
        try:
            n1 = float(val1)
            n2 = float(val2)
            return 1.0 if abs(n1 - n2) < 0.01 else 0.0
        except:
            return 0.0

    return 1.0 if str(val1).strip() == str(val2).strip() else 0.0

def compute_outbreak_similarity(perg_row: pd.Series, extracted_row: pd.Series) -> float:
    weights = normalise_weights(OUTBREAK_HIGH_WEIGHT_FIELDS)
    sim = 0.0
    for field, w in weights.items():
        if field not in perg_row.index or field not in extracted_row.index:
            continue
        sim += w * compute_outbreak_field_similarity(perg_row[field], extracted_row[field], field)
    return sim

def match_outbreaks_in_article(perg_outbreaks: pd.DataFrame, extracted_outbreaks: pd.DataFrame):
    n_perg = len(perg_outbreaks)
    n_extracted = len(extracted_outbreaks)
    if n_perg == 0 or n_extracted == 0:
        return []

    similarity_matrix = np.zeros((n_perg, n_extracted))
    for i, (_, perg_row) in enumerate(perg_outbreaks.iterrows()):
        for j, (_, extracted_row) in enumerate(extracted_outbreaks.iterrows()):
            similarity_matrix[i, j] = compute_outbreak_similarity(perg_row, extracted_row)

    cost_matrix = 1 - similarity_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    return [
        {'perg_idx': perg_outbreaks.index[i], 'extracted_idx': extracted_outbreaks.index[j], 'similarity': similarity_matrix[i, j]}
        for i, j in zip(row_ind, col_ind)
    ]

def evaluate_outbreak_count_optimal(perg: pd.DataFrame, extracted: pd.DataFrame, common_articles: list):
    tp_total = fp_total = fn_total = 0
    for article_id in common_articles:
        perg_count = len(perg[perg['article_id'] == article_id])
        extracted_count = len(extracted[extracted['article_id'] == article_id])
        tp_total += min(perg_count, extracted_count)
        fp_total += max(0, extracted_count - perg_count)
        fn_total += max(0, perg_count - extracted_count)

    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3)}

def evaluate_outbreak_field_optimal(perg: pd.DataFrame, extracted: pd.DataFrame, common_articles: list, field_name: str):
    tp_total = fp_total = fn_total = 0

    for article_id in common_articles:
        perg_outbreaks = perg[perg['article_id'] == article_id]
        extracted_outbreaks = extracted[extracted['article_id'] == article_id]
        matches = match_outbreaks_in_article(perg_outbreaks, extracted_outbreaks)

        for match in matches:
            perg_val = perg.loc[match['perg_idx'], field_name]
            extracted_val = extracted.loc[match['extracted_idx'], field_name]

            if field_name in ['asymptomatic_transmission_described', 'outbreak_is_currently_ongoing']:
                p = normalize_boolean(perg_val)
                e = normalize_boolean(extracted_val)
                if p is None and e is None:
                    tp_total += 1
                elif p is None or e is None:
                    if e is not None: fp_total += 1
                    if p is not None: fn_total += 1
                elif p == e:
                    tp_total += 1
                else:
                    fp_total += 1
                    fn_total += 1

            elif field_name in ['outbreak_start_day', 'outbreak_end_day', 'outbreak_start_year', 'outbreak_end_year',
                                'cases_confirmed', 'deaths']:
                if pd.isna(perg_val) and pd.isna(extracted_val):
                    tp_total += 1
                elif pd.isna(perg_val) or pd.isna(extracted_val):
                    if not pd.isna(extracted_val): fp_total += 1
                    if not pd.isna(perg_val): fn_total += 1
                else:
                    try:
                        if abs(float(perg_val) - float(extracted_val)) < 0.01:
                            tp_total += 1
                        else:
                            fp_total += 1
                            fn_total += 1
                    except:
                        fp_total += 1
                        fn_total += 1
            else:
                if pd.isna(perg_val) and pd.isna(extracted_val):
                    tp_total += 1
                elif pd.isna(perg_val) or pd.isna(extracted_val):
                    if not pd.isna(extracted_val): fp_total += 1
                    if not pd.isna(perg_val): fn_total += 1
                elif str(perg_val).strip() == str(extracted_val).strip():
                    tp_total += 1
                else:
                    fp_total += 1
                    fn_total += 1

    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return {'precision': round(precision, 3), 'recall': round(recall, 3), 'f1': round(f1, 3)}

def evaluate_outbreak_pathogen(pathogen: str, method='optimal', filter_invalid: bool = True, perg_extractions_path=None, extracted_extractions_path=None):
    perg, extracted, common_articles, filter_stats = load_outbreak_data(
        pathogen, filter_invalid, perg_extractions_path, extracted_extractions_path
    )

    results = []
    count_metrics = evaluate_outbreak_count_optimal(perg, extracted, common_articles)
    results.append({'pathogen': pathogen, 'field': 'outbreak_count', **count_metrics})

    fields_to_evaluate = [
        ('outbreak_start_day', False),
        ('outbreak_start_month', False),
        ('outbreak_start_year', False),
        ('outbreak_end_day', False),
        ('outbreak_end_month', False),
        ('outbreak_end_year', False),
        ('cases_confirmed', False),
        ('deaths', False),
        ('outbreak_country', False),
        ('outbreak_location', False),
        ('mode_of_detection', False),
        ('pre_outbreak', False),
        ('outbreak_is_currently_ongoing', False),
        ('asymptomatic_transmission_described', False),
    ]

    for field_name, _ in fields_to_evaluate:
        if field_name not in perg.columns or field_name not in extracted.columns:
            continue
        metrics = evaluate_outbreak_field_optimal(perg, extracted, common_articles, field_name)
        results.append({'pathogen': pathogen, 'field': field_name, **metrics})

    return results, filter_stats

In [63]:
def compute_outbreak_flagging_metrics(
    pathogen,
    model_name,
    fulltext_screening_path,
    perg_screening_path,
    outbreaks_extraction_path,
    perg_outbreaks_path
):
    perg_screening_path = perg_screening_files[pathogen]
    fulltext_screening_path = fulltext_screening_agentslr[(pathogen, model_name)]
    outbreaks_extraction_path = outbreaks_agentslr[(pathogen, model_name)]
    perg_outbreaks_path = perg_outbreaks_files[pathogen]

    df_fulltext = pd.read_csv(fulltext_screening_path)
    df_perg_screening = pd.read_csv(perg_screening_path)
    df_outbreaks = pd.read_csv(outbreaks_extraction_path)
    df_perg_outbreaks = pd.read_csv(perg_outbreaks_path)

    cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']
    for col in cols_to_be_added:
        if col in df_fulltext.columns:
            df_fulltext = df_fulltext.drop(columns=col)

    df_fulltext = add_perg_columns_from_screening(df_fulltext, df_perg_screening)

    df_relevant = df_fulltext[df_fulltext['perg_fulltext_result'] == 'INCLUDE'].copy()

    df_relevant['covidence_id'] = df_relevant['Covidence #'].apply(
        lambda x: x.replace('#', '').strip() if pd.notnull(x) else x
    )

    flagged_positive_cov_id = set(map(int, df_outbreaks['article_id'].dropna().unique()))

    true_all_perg = set(map(int, df_perg_outbreaks['covidence_id'].dropna().unique()))

    df_relevant['screening_ai_flag'] = df_relevant['covidence_id'].apply(
        lambda v: True if pd.notnull(v) and int(str(v).replace('#', '').strip()) in flagged_positive_cov_id else False
    )

    df_relevant['true_perg_flag'] = df_relevant['covidence_id'].apply(
        lambda v: True if pd.notnull(v) and int(str(v).replace('#', '').strip()) in true_all_perg else False
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        df_relevant["true_perg_flag"],
        df_relevant["screening_ai_flag"],
        average='macro',
    )

    return {
            'pathogen': pathogen,
            'model': model_name,
            'field': 'Article Flagging',
            'precision': round(precision, 3),
            'recall': round(recall, 3),
            'f1': round(f1, 3),
            'n_articles': len(df_relevant)
        }

## Outbreak Extraction Results

In [64]:
METHOD = "gpt"
FILTER_INVALID = True
model_name = 'deepseek'

all_results = []
all_filter_stats = []

for pathogen in ['Lassa', 'Zika']:
    print(f"Evaluating {pathogen}...")
    perg = f'../data/perg/extracted/{pathogen.lower()}_outbreaks.csv'
    extracted = f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_outbreaks.csv'

    pathogen_results, filter_stats = evaluate_outbreak_pathogen(
        pathogen, method=METHOD, filter_invalid=FILTER_INVALID,
        perg_extractions_path=perg, extracted_extractions_path=extracted
    )
    
    all_results.extend(pathogen_results)
    all_filter_stats.append(filter_stats)

results_df = pd.DataFrame(all_results)
filter_stats_df = pd.DataFrame(all_filter_stats)

Evaluating Lassa...
Evaluating Zika...


In [65]:
filter_stats_df

,pathogen,perg_total,perg_invalid,perg_invalid_pct,extracted_total,extracted_invalid,extracted_invalid_pct
0,Lassa,30,0,0.00,33,0,0.0
1,Zika,159,15,9.43,69,0,0.0


In [66]:
results_df

,pathogen,field,precision,recall,f1
0,Lassa,outbreak_count,0.833,1.000,0.909
1,Lassa,outbreak_start_day,0.800,1.000,0.889
2,Lassa,outbreak_start_month,0.800,0.800,0.800
3,Lassa,outbreak_start_year,0.800,0.800,0.800
4,Lassa,outbreak_end_day,0.400,0.500,0.444
5,Lassa,outbreak_end_month,0.800,0.800,0.800
6,Lassa,cases_confirmed,0.800,0.800,0.800
7,Lassa,deaths,1.000,1.000,1.000
8,Lassa,outbreak_country,1.000,1.000,1.000
9,Lassa,outbreak_location,0.000,0.000,0.000


In [67]:
model_name = 'oss'

flagging_results = []

for pathogen in ['Lassa', 'Zika']:

    metrics = compute_outbreak_flagging_metrics(
        pathogen,
        model_name,
        fulltext_screening_path=fulltext_screening_agentslr[(pathogen, model_name)],
        perg_screening_path=perg_screening_files[pathogen],
        outbreaks_extraction_path=outbreaks_agentslr[(pathogen, model_name)],
        perg_outbreaks_path=perg_outbreaks_files[pathogen]
    )
    
    flagging_results.append(metrics)

In [68]:
flagging_results_df = pd.DataFrame(flagging_results)
flagging_results_df

,pathogen,model,field,precision,recall,f1,n_articles
0,Lassa,oss,Article Flagging,0.690,0.816,0.705,49
1,Zika,oss,Article Flagging,0.576,0.713,0.525,166


In [69]:
def create_aggregated_summary_table(results_df, article_flagging_metrics=None):
    pathogens = sorted(results_df['pathogen'].unique())
    
    categories = {
        'Temporal': ['outbreak_start_day', 'outbreak_start_month', 'outbreak_start_year',
                     'outbreak_end_day', 'outbreak_end_month', 'outbreak_end_year'],
        'Geographic': ['outbreak_country', 'outbreak_location'],
        'Case Burden': ['cases_confirmed', 'deaths'],
        'Epidemiological': ['mode_of_detection', 'pre_outbreak_status']
    }
    
    summary_data = []
    
    if article_flagging_metrics is not None:
        flag_row = {'Field': 'Article Flagging'}
        for pathogen in pathogens:
            pathogen_data = article_flagging_metrics[article_flagging_metrics['pathogen'] == pathogen]
            if len(pathogen_data) > 0:
                flag_row[f"{pathogen}_P"] = pathogen_data.iloc[0]['precision']
                flag_row[f"{pathogen}_R"] = pathogen_data.iloc[0]['recall']
                flag_row[f"{pathogen}_F1"] = pathogen_data.iloc[0]['f1']
            else:
                flag_row[f"{pathogen}_P"] = np.nan
                flag_row[f"{pathogen}_R"] = np.nan
                flag_row[f"{pathogen}_F1"] = np.nan
        summary_data.append(flag_row)
    
    count_row = {'Field': 'Outbreak Counts'}
    for pathogen in pathogens:
        count_data = results_df[(results_df['pathogen'] == pathogen) & (results_df['field'] == 'outbreak_count')]
        if len(count_data) > 0:
            count_row[f"{pathogen}_P"] = count_data.iloc[0]['precision']
            count_row[f"{pathogen}_R"] = count_data.iloc[0]['recall']
            count_row[f"{pathogen}_F1"] = count_data.iloc[0]['f1']
        else:
            count_row[f"{pathogen}_P"] = np.nan
            count_row[f"{pathogen}_R"] = np.nan
            count_row[f"{pathogen}_F1"] = np.nan
    summary_data.append(count_row)
    
    for category_name, fields in categories.items():
        category_row = {'Field': category_name}
        for pathogen in pathogens:
            pathogen_data = results_df[(results_df['pathogen'] == pathogen) & (results_df['field'].isin(fields))]
            if len(pathogen_data) > 0:
                p_mean = pathogen_data['precision'].mean()
                r_mean = pathogen_data['recall'].mean()
                category_row[f"{pathogen}_P"] = round(p_mean, 3)
                category_row[f"{pathogen}_R"] = round(r_mean, 3)
                category_row[f"{pathogen}_F1"] = round(_f1_from_pr(p_mean, r_mean), 3)
            else:
                category_row[f"{pathogen}_P"] = np.nan
                category_row[f"{pathogen}_R"] = np.nan
                category_row[f"{pathogen}_F1"] = np.nan
        summary_data.append(category_row)
    
    overall_row = {'Field': 'Overall (Extraction)'}
    for pathogen in pathogens:
        pathogen_data = results_df[results_df['pathogen'] == pathogen]
        extraction_data = pathogen_data[~pathogen_data['field'].isin(['outbreak_count'])]
        
        p_mean = extraction_data['precision'].mean()
        r_mean = extraction_data['recall'].mean()
        
        overall_row[f"{pathogen}_P"] = round(p_mean, 3)
        overall_row[f"{pathogen}_R"] = round(r_mean, 3)
        overall_row[f"{pathogen}_F1"] = round(_f1_from_pr(p_mean, r_mean), 3)
    summary_data.append(overall_row)
    
    return pd.DataFrame(summary_data)

aggregated_summary = create_aggregated_summary_table(results_df)

In [70]:
aggregated_summary

,Field,Lassa_P,Lassa_R,Lassa_F1,Zika_P,Zika_R,Zika_F1
0,Outbreak Counts,0.833,1.000,0.909,0.680,0.607,0.642
1,Temporal,0.720,0.780,0.749,0.694,0.903,0.785
2,Geographic,0.500,0.500,0.500,0.588,0.613,0.600
3,Case Burden,0.900,0.900,0.900,0.912,0.966,0.938
4,Epidemiological,0.600,0.600,0.600,0.412,0.412,0.412
5,Overall (Extraction),0.754,0.777,0.765,0.692,0.805,0.744


## Outbreak Extraction Ablation Study

In [71]:
models = ['oss', 'deepseek', 'kimi', 'glm', 'gpt']
df_all = []
METHOD = "optimal"
FILTER_INVALID = True

for model_name in tqdm(models, desc="Evaluating models"):
    all_results = []
    
    for pathogen in ['Lassa', 'Zika']:
        perg = f'../data/perg/extracted/{pathogen.lower()}_outbreaks.csv'
        extracted = f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_outbreaks.csv'
        
        pathogen_results, _ = evaluate_outbreak_pathogen(
            pathogen,
            method=METHOD,
            filter_invalid=FILTER_INVALID,
            perg_extractions_path=perg,
            extracted_extractions_path=extracted
        )
        
        for result in pathogen_results:
            result['model'] = model_name
        
        all_results.extend(pathogen_results)

        metrics = compute_outbreak_flagging_metrics(
            pathogen,
            model_name,
            fulltext_screening_path=fulltext_screening_agentslr[(pathogen, model_name)],
            perg_screening_path=perg_screening_files[pathogen],
            outbreaks_extraction_path=outbreaks_agentslr[(pathogen, model_name)],
            perg_outbreaks_path=perg_outbreaks_files[pathogen]
        )

        all_results.append({
            "pathogen": pathogen,
            "model": model_name,
            "field": "Article Flagging",
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1": metrics["f1"],
        })
    
    df_all.append(pd.DataFrame(all_results))

df_outbreak_extractions_all = pd.concat(df_all, ignore_index=True)

Evaluating models: 100%|██████████| 5/5 [00:16<00:00,  3.38s/it]


In [72]:
df_outbreak_extractions_all

,pathogen,field,precision,recall,f1,model
0,Lassa,outbreak_count,0.833,1.000,0.909,oss
1,Lassa,outbreak_start_day,0.857,0.667,0.750,oss
2,Lassa,outbreak_start_month,0.778,0.778,0.778,oss
3,Lassa,outbreak_start_year,0.889,0.800,0.842,oss
4,Lassa,outbreak_end_day,0.857,0.667,0.750,oss
...,...,...,...,...,...,...
145,Zika,mode_of_detection,0.800,0.800,0.800,gpt
146,Zika,pre_outbreak,0.182,0.051,0.080,gpt
147,Zika,outbreak_is_currently_ongoing,0.978,0.978,0.978,gpt
148,Zika,asymptomatic_transmission_described,1.000,1.000,1.000,gpt


In [73]:
df_outbreak_extractions_all.to_csv('../data/agentslr/evals/data_extraction/outbreaks/all_outbreak_extractions_results.csv', index=False)

In [74]:
records = []
for model_name, model_df in df_outbreak_extractions_all.groupby('model'):
    avg_precision = model_df['precision'].mean()
    avg_recall = model_df['recall'].mean()
    avg_f1 = model_df['f1'].mean()
    
    records.append({
        'model': model_name,
        'avg_precision': round(avg_precision, 3),
        'avg_recall': round(avg_recall, 3),
        'avg_f1': round(avg_f1, 3)
    })

pd.DataFrame(records)

,model,avg_precision,avg_recall,avg_f1
0,deepseek,0.719,0.788,0.744
1,glm,0.784,0.758,0.762
2,gpt,0.835,0.836,0.827
3,kimi,0.770,0.813,0.777
4,oss,0.819,0.755,0.769


## Save Outbreak Extraction CSVs

In [75]:
save_all_extraction_metrics(df_outbreak_extractions_all, 'outbreaks', EVALS_OUTBREAKS_DIR)
print("Outbreak extraction metrics saved!")

Outbreak extraction metrics saved!


---

# Parameter Extraction Evaluation

## Parameter Schema Definition

Parameter extraction differs from models and outbreaks in calculating Flagging and Count metrics at the level of parameter **classes**. While models and outbreaks have only one data_type, parameters considers one data_type per schematised parameter class.

We define key parameter fields as:

$$\mathcal{F}_{\text{params}} = \{\texttt{parameter\_class}, \texttt{parameter\_type}, \texttt{value}, \texttt{unit}, \texttt{method}, \texttt{value\_type}, \texttt{statistical\_approach}, \texttt{paired\_uncertainty}, \texttt{single\_type\_uncertainty}, \texttt{population\_sex}, \texttt{population\_group}, \texttt{population\_sample\_type}\}$$

Fields are grouped by sub-stage to ensure equal importance in determining similarity:
- **Categorical fields** (2): parameter class, parameter type
- **Value fields** (3): value, unit, method
- **Uncertainty fields** (4): value type, statistical approach, single type uncertainty, paired uncertainty
- **Population fields** (3): population sex, population group, population sample type

## Parameter Data Preparation

In [33]:
UNIT_VALUES = ['days', 'weeks', 'months', 'years', 'hours', 'per_day', 'per_week', 'per_month', 'per_year', 'percentage', 'unspecified']

PROPORTION_TO_PERCENTAGE_CLASSES = {'seroprevalence', 'severity'}

In [34]:
pathogens = ["Ebola", "Lassa", "SARS", "Zika"]
models = ["oss", "deepseek", "kimi", "glm", "gpt"]

### Parameter File Paths and Article ID Mapping

In [35]:
parameters_agentslr = {
    (pathogen, model_name): f'../data/agentslr/client/{model_name}/{pathogen.lower()}/extractions/data_extraction_parameters.jsonl' for pathogen in pathogens for model_name in models
}

original_screening_files = {
    (pathogen, model_name): f'../data/agentslr/client/{model_name}/{pathogen.lower()}/screening/fulltext_screening_results.csv' for pathogen in pathogens for model_name in models
}

perg_screening_files = {
    pathogen: f'../data/perg/screening/{pathogen}_filtered.csv' for pathogen in pathogens
}

perg_parameters_files = {
    pathogen: f"../data/perg/extracted/{pathogen.lower()}_parameters.csv"
    for pathogen in pathogens
}

In [36]:

pathogens = ["Ebola", "Lassa", "SARS", "Zika"]
models = ["oss", "deepseek", "kimi", "glm", "gpt"]

root = Path("../data/agentslr/client")
prefix = "data_extraction_parameters_dumps_"
ts_re = re.compile(rf"^{re.escape(prefix)}(\d{{4}}-\d{{2}}-\d{{2}}_\d{{2}}-\d{{2}}-\d{{2}})$")

parameters_flagging_agentslr = {}

for pathogen in pathogens:
    for model_name in models:
        base = root / model_name / pathogen / "logs"
        candidates = []

        if base.exists():
            for d in base.iterdir():
                if d.is_dir():
                    m = ts_re.match(d.name)
                    if m:
                        candidates.append((m.group(1), d))

        candidates.sort(key=lambda x: x[0], reverse=True)

        chosen = None
        missing = []

        for ts, d in candidates:
            target = d / "results" / "screening.jsonl"
            if target.is_file():
                chosen = target
                break
            missing.append(ts)

        key = (pathogen, model_name)

        if chosen is None:
            parameters_flagging_agentslr[key] = None
            if candidates:
                print(f"MISSING ALL: {key} checked={ [ts for ts, _ in candidates] }")
            else:
                print(f"MISSING LOGS DIR: {key}")
        else:
            parameters_flagging_agentslr[key] = str(chosen)
            if missing:
                print(
                    f"FALLBACK USED: {key} "
                    f"used={chosen.parent.parent.name} "
                    f"missing_newer={missing}"
                )



FALLBACK USED: ('Lassa', 'gpt') used=data_extraction_parameters_dumps_2026-02-10_18-53-33 missing_newer=['2026-02-13_16-17-36']


In [37]:
parameters_flagging_agentslr

{('Ebola',
  'oss'): '../data/agentslr/client/oss/Ebola/logs/data_extraction_parameters_dumps_2026-02-06_15-35-58/results/screening.jsonl',
 ('Ebola',
  'deepseek'): '../data/agentslr/client/deepseek/Ebola/logs/data_extraction_parameters_dumps_2026-02-07_21-52-23/results/screening.jsonl',
 ('Ebola',
  'kimi'): '../data/agentslr/client/kimi/Ebola/logs/data_extraction_parameters_dumps_2026-02-08_00-42-41/results/screening.jsonl',
 ('Ebola',
  'glm'): '../data/agentslr/client/glm/Ebola/logs/data_extraction_parameters_dumps_2026-02-09_02-49-48/results/screening.jsonl',
 ('Ebola',
  'gpt'): '../data/agentslr/client/gpt/Ebola/logs/data_extraction_parameters_dumps_2026-02-13_23-34-55/results/screening.jsonl',
 ('Lassa',
  'oss'): '../data/agentslr/client/oss/Lassa/logs/data_extraction_parameters_dumps_2026-02-06_13-35-12/results/screening.jsonl',
 ('Lassa',
  'deepseek'): '../data/agentslr/client/deepseek/Lassa/logs/data_extraction_parameters_dumps_2026-02-08_07-32-46/results/screening.jsonl'

In [38]:
for model_name in models:
    print(f"\n\nMODEL: {model_name}")
    for pathogen in pathogens:
        print(f"Processing {pathogen}...")

        df_agentslr_fulltext_screening = pd.read_csv(original_screening_files[(pathogen, model_name)])
        df_perg = pd.read_csv(perg_screening_files[pathogen])

        cols_to_be_added = ['perg_fulltext_result', 'perg_abstract_result', 'Covidence #', 'perg_subset']

        for col in cols_to_be_added:
            if col in df_agentslr_fulltext_screening.columns:
                df_agentslr_fulltext_screening = df_agentslr_fulltext_screening.drop(columns=col)

        df_agentslr_fulltext_screening = add_perg_columns_from_screening(df_agentslr_fulltext_screening, df_perg)

        df_agentslr_fulltext_screening['covidence_id'] = df_agentslr_fulltext_screening['Covidence #'].apply(
            lambda x: x.replace('#', '').strip() if pd.notnull(x) else x
        )

        print(
            f"\nNumber of articles conditioned on PERG for {pathogen} and model {model_name}:",
            len(df_agentslr_fulltext_screening[df_agentslr_fulltext_screening.perg_fulltext_result == 'INCLUDE'])
        )

        df_agentslr_params = pd.read_json(parameters_agentslr[(pathogen, model_name)], lines=True)
        df_agentslr_params_flagging = pd.read_json(parameters_flagging_agentslr[(pathogen, model_name)], lines=True)

        if 'article_id' in df_agentslr_params.columns and 'article_uuid' in df_agentslr_params.columns:
            print(f"Both article_id and article_uuid are present in df_agentslr_params for {pathogen} and model {model_name}. Skipping the merging process for extraction parameters.")
        else:
            df_agentslr_params = df_agentslr_params[[col for col in df_agentslr_params.columns if 'covidence' not in col.lower()]]

            df_merged = df_agentslr_params.merge(
                df_agentslr_fulltext_screening[['article_id', 'covidence_id']],
                on='article_id',
                how='left'
            )

            df_merged = df_merged.rename(columns={'article_id': 'article_uuid'})
            df_merged = df_merged.rename(columns={'covidence_id': 'article_id'})

            print(f"Number of unique articles after merging with covidence_id (extraction params):", len(df_merged.article_id.unique()))

            df_merged.to_json(parameters_agentslr[(pathogen, model_name)], orient="records", lines=True, date_format="iso")

        if 'article_id' in df_agentslr_params_flagging.columns and 'article_uuid' in df_agentslr_params_flagging.columns:
            print(f"Both article_id and article_uuid are present in df_agentslr_params_flagging for {pathogen} and model {model_name}. Skipping the merging process for flagging parameters.")
        else:
            df_agentslr_params_flagging = df_agentslr_params_flagging[[col for col in df_agentslr_params_flagging.columns if 'covidence' not in col.lower()]]

            df_merged_flagging = df_agentslr_params_flagging.merge(
                df_agentslr_fulltext_screening[['article_id', 'covidence_id']],
                on='article_id',
                how='left'
            )

            df_merged_flagging = df_merged_flagging.rename(columns={'article_id': 'article_uuid'})
            df_merged_flagging = df_merged_flagging.rename(columns={'covidence_id': 'article_id'})

            print(f"Number of unique articles after merging with covidence_id (flagging params):", len(df_merged_flagging.article_id.unique()))

            df_merged_flagging.to_json(parameters_flagging_agentslr[(pathogen, model_name)], orient="records", lines=True, date_format="iso")



MODEL: oss
Processing Ebola...

Number of articles conditioned on PERG for Ebola and model oss: 223
Both article_id and article_uuid are present in df_agentslr_params for Ebola and model oss. Skipping the merging process for extraction parameters.
Both article_id and article_uuid are present in df_agentslr_params_flagging for Ebola and model oss. Skipping the merging process for flagging parameters.
Processing Lassa...

Number of articles conditioned on PERG for Lassa and model oss: 49
Both article_id and article_uuid are present in df_agentslr_params for Lassa and model oss. Skipping the merging process for extraction parameters.
Both article_id and article_uuid are present in df_agentslr_params_flagging for Lassa and model oss. Skipping the merging process for flagging parameters.
Processing SARS...

Number of articles conditioned on PERG for SARS and model oss: 103
Both article_id and article_uuid are present in df_agentslr_params for SARS and model oss. Skipping the merging proce

## Parameter Field Pre-processing and Mappings

In [39]:
PERG_FIELD_MAPPING = {
    'parameter_unit': 'unit',
    'parameter_value': 'value',
    'parameter_value_type': 'value_type',
    'cfr_ifr_method': 'method',
    'parameter_statistical_approach': 'statistical_approach',
    'parameter_uncertainty_type': 'paired_uncertainty',
    'parameter_uncertainty_singe_type': 'single_type_uncertainty',
    'population_sex': 'population_sex',
    'population_group': 'population_group',
    'population_sample_type': 'population_sample_type',
    'covidence_id': 'article_id',
}

PERG_VALUE_MAPPINGS = {
    'unit': PARAMETER_UNITS_MAPPING,
    'method': METHOD_MAPPING,
    'value_type': VALUE_TYPE_MAPPING,
    'statistical_approach': STATISTICAL_APPROACH_MAPPING,
    'paired_uncertainty': PAIRED_UNCERTAINTY_MAPPING,
    'single_type_uncertainty': SINGLE_TYPE_UNCERTAINTY_MAPPING,
    'population_sex': POPULATION_SEX_MAPPING,
    'population_group': POPULATION_GROUP_MAPPING,
    'population_sample_type': POPULATION_SAMPLE_TYPE_MAPPING,
}

PARAMETER_FIELD_VALIDATORS = {
    'unit': (UNIT_VALUES, False),
}

PARAMETER_HIGH_WEIGHT_FIELDS = {
    'parameter_class': 1,
    'unit': 1,
}

NUMERIC_FIELDS = {'value'}

PARAMETER_FIELD_DISPLAY_NAMES = {
    'flagging': 'Article Flagging',
    'parameter_count': 'Parameter Counts',
    'unit': 'Unit',
    'value': 'Value',
    'method': 'Method',
    'value_type': 'Value Type',
    'statistical_approach': 'Statistical Approach',
    'paired_uncertainty': 'Paired Uncertainty',
    'single_type_uncertainty': 'Single Type Uncertainty',
    'population_sex': 'Population Sex',
    'population_group': 'Population Group',
    'population_sample_type': 'Population Sample Type',
}

In [40]:
def normalise_weights(weights):
    total = sum(weights.values())
    return {k: v/total for k, v in weights.items()}

def is_valid_parameter_field_value(val, valid_values, is_multivalue):
    if pd.isna(val):
        return True
    
    valid_set = set(valid_values)
    
    if is_multivalue:
        items = [item.strip() for item in str(val).split(';') if item.strip()]
        return all(item in valid_set for item in items)
    else:
        return str(val).strip() in valid_set

def filter_invalid_parameter_rows(df, pathogen_name, filter_invalid=True):
    total_rows = len(df)
    
    if not filter_invalid:
        return df, 0, 0.0
    
    invalid_mask = pd.Series([False] * len(df), index=df.index)
    
    for field_name, (valid_values, is_multivalue) in PARAMETER_FIELD_VALIDATORS.items():
        if field_name not in df.columns:
            continue
        
        for idx, row in df.iterrows():
            val = row[field_name]
            if not is_valid_parameter_field_value(val, valid_values, is_multivalue):
                invalid_mask[idx] = True
    
    invalid_count = invalid_mask.sum()
    invalid_pct = (invalid_count / total_rows * 100) if total_rows > 0 else 0
    
    df_filtered = df[~invalid_mask].copy()
    
    return df_filtered, invalid_count, invalid_pct

def compare_numeric_values(val1, val2, tolerance=1e-9):
    if pd.isna(val1) and pd.isna(val2):
        return True
    if pd.isna(val1) or pd.isna(val2):
        return False
    try:
        num1 = float(val1)
        num2 = float(val2)
        return abs(num1 - num2) <= tolerance
    except (ValueError, TypeError):
        return False

def normalise_extracted_values(df):
    if 'value' not in df.columns or 'parameter_class' not in df.columns:
        return df
    
    df = df.copy()
    mask = df['parameter_class'].isin(PROPORTION_TO_PERCENTAGE_CLASSES)
    
    for idx in df[mask].index:
        val = df.loc[idx, 'value']
        if pd.notna(val):
            try:
                num_val = float(val)
                if 0 <= num_val <= 1:
                    df.loc[idx, 'value'] = num_val * 100
            except (ValueError, TypeError):
                pass
    
    return df

In [41]:
def load_jsonl_extractions(filepath):
    import json
    
    records = []
    with open(filepath, 'r') as f:
        for line in f:
            if line.strip():
                data = json.loads(line)
                record = {
                    'article_id': data.get('article_id'),
                    'parameter_class': data.get('parameter_class'),
                }
                if 'extraction' in data and data['extraction']:
                    for key, value in data['extraction'].items():
                        if key not in ['article_id', 'parameter_class']:
                            record[key] = value
                records.append(record)
    
    return pd.DataFrame(records)

def load_perg_parameters(filepath):
    df = pd.read_csv(filepath)
    
    df = df.rename(columns=PERG_FIELD_MAPPING)
    
    for field_name, mapping in PERG_VALUE_MAPPINGS.items():
        if field_name in df.columns:
            df[field_name] = df[field_name].map(mapping).fillna(df[field_name])
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    return df

def load_parameter_data(agentslr_params_file, perg_params_file, pathogen, filter_invalid=True):
    perg = load_perg_parameters(perg_params_file)
    
    extracted = load_jsonl_extractions(agentslr_params_file)
    
    perg = perg.loc[:, ~perg.columns.duplicated()]
    extracted = extracted.loc[:, ~extracted.columns.duplicated()]
    
    perg['article_id'] = perg['article_id'].astype(str)
    extracted['article_id'] = extracted['article_id'].astype(str)
    
    if 'parameter_class' in perg.columns:
        perg['parameter_class'] = perg['parameter_class'].map(PARAMETER_CLASSES_MAPPING).fillna(perg['parameter_class'])
    
    if 'parameter_class' in perg.columns:
        perg['parameter_class'] = perg['parameter_class'].str.lower().str.replace(' ', '_')
    if 'parameter_class' in extracted.columns:
        extracted['parameter_class'] = extracted['parameter_class'].str.lower().str.replace(' ', '_')
    
    extracted = normalise_extracted_values(extracted)
    
    perg_filtered, perg_invalid_count, perg_invalid_pct = filter_invalid_parameter_rows(perg, pathogen, filter_invalid)
    extracted_filtered, extracted_invalid_count, extracted_invalid_pct = filter_invalid_parameter_rows(extracted, pathogen, filter_invalid)
    
    perg_article_ids = set(perg_filtered['article_id'].dropna().tolist())
    extracted_article_ids = set(extracted_filtered['article_id'].dropna().tolist())
    common_articles = perg_article_ids & extracted_article_ids
    
    perg_filtered = perg_filtered[perg_filtered['article_id'].isin(common_articles)]
    extracted_filtered = extracted_filtered[extracted_filtered['article_id'].isin(common_articles)]
    
    filter_stats = {
        'pathogen': pathogen,
        'perg_total': len(perg),
        'perg_invalid': perg_invalid_count,
        'perg_invalid_pct': round(perg_invalid_pct, 2),
        'extracted_total': len(extracted),
        'extracted_invalid': extracted_invalid_count,
        'extracted_invalid_pct': round(extracted_invalid_pct, 2),
        'common_articles': len(common_articles)
    }
    
    return perg_filtered, extracted_filtered, list(common_articles), filter_stats

In [42]:
def compute_parameter_similarity(perg_row, extracted_row):
    weights = normalise_weights(PARAMETER_HIGH_WEIGHT_FIELDS)
    
    similarity = 0.0
    for field, weight in weights.items():
        if field not in perg_row.index or field not in extracted_row.index:
            continue
        
        perg_val = perg_row[field]
        extracted_val = extracted_row[field]
        
        if pd.isna(perg_val) and pd.isna(extracted_val):
            field_sim = 1.0
        elif pd.isna(perg_val) or pd.isna(extracted_val):
            field_sim = 0.0
        else:
            field_sim = 1.0 if str(perg_val).strip() == str(extracted_val).strip() else 0.0
        
        similarity += weight * field_sim
    
    return similarity

def match_parameters_in_article(perg_params, extracted_params):
    n_perg = len(perg_params)
    n_extracted = len(extracted_params)
    
    if n_perg == 0 or n_extracted == 0:
        return []
    
    similarity_matrix = np.zeros((n_perg, n_extracted))
    
    for i, (_, perg_row) in enumerate(perg_params.iterrows()):
        for j, (_, extracted_row) in enumerate(extracted_params.iterrows()):
            similarity_matrix[i, j] = compute_parameter_similarity(perg_row, extracted_row)
    
    cost_matrix = 1 - similarity_matrix
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    matches = []
    for i, j in zip(row_ind, col_ind):
        matches.append({
            'perg_idx': perg_params.index[i],
            'extracted_idx': extracted_params.index[j],
            'similarity': similarity_matrix[i, j]
        })
    
    return matches

def evaluate_parameter_count_optimal(perg, extracted, common_articles):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_count = len(perg[perg['article_id'] == article_id])
        extracted_count = len(extracted[extracted['article_id'] == article_id])
        
        tp = min(perg_count, extracted_count)
        fp = max(0, extracted_count - perg_count)
        fn = max(0, perg_count - extracted_count)
        
        tp_total += tp
        fp_total += fp
        fn_total += fn
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

def evaluate_parameter_field_optimal(perg, extracted, common_articles, field_name, is_multivalue, is_numeric=False):
    tp_total = 0
    fp_total = 0
    fn_total = 0
    
    for article_id in common_articles:
        perg_params = perg[perg['article_id'] == article_id]
        extracted_params = extracted[extracted['article_id'] == article_id]
        
        matches = match_parameters_in_article(perg_params, extracted_params)
        
        for match in matches:
            perg_val = perg.loc[match['perg_idx'], field_name] if field_name in perg.columns else None
            extracted_val = extracted.loc[match['extracted_idx'], field_name] if field_name in extracted.columns else None
            
            if is_numeric:
                if compare_numeric_values(perg_val, extracted_val):
                    tp_total += 1
                else:
                    fp_total += 1
                    fn_total += 1
            elif is_multivalue:
                perg_set = (
                    {t.strip() for t in str(perg_val).split(';') if t.strip()}
                    if perg_val is not None and not pd.isna(perg_val)
                    else set()
                )
                extracted_set = (
                    {t.strip() for t in str(extracted_val).split(';') if t.strip()}
                    if extracted_val is not None and not pd.isna(extracted_val)
                    else set()
                )
                
                tp = len(perg_set & extracted_set)
                fp = len(extracted_set - perg_set)
                fn = len(perg_set - extracted_set)
                
                tp_total += tp
                fp_total += fp
                fn_total += fn
            else:
                if (perg_val is None or pd.isna(perg_val)) and (extracted_val is None or pd.isna(extracted_val)):
                    tp_total += 1
                elif perg_val is None or pd.isna(perg_val) or extracted_val is None or pd.isna(extracted_val):
                    if extracted_val is not None and not pd.isna(extracted_val):
                        fp_total += 1
                    if perg_val is not None and not pd.isna(perg_val):
                        fn_total += 1
                elif str(perg_val).strip() == str(extracted_val).strip():
                    tp_total += 1
                else:
                    fp_total += 1
                    fn_total += 1
    
    precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0
    recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return {
        'precision': round(precision, 3),
        'recall': round(recall, 3),
        'f1': round(f1, 3)
    }

In [43]:
def evaluate_parameter_pathogen(pathogen, model_name, parameters_agentslr, perg_parameters_files, filter_invalid=True):
    agentslr_params_file = parameters_agentslr[(pathogen, model_name)]
    perg_params_file = perg_parameters_files[pathogen]
    
    perg, extracted, common_articles, filter_stats = load_parameter_data(
        agentslr_params_file, perg_params_file, pathogen, filter_invalid
    )
    
    filter_stats['model'] = model_name
    
    results = []
    
    param_count_metrics = evaluate_parameter_count_optimal(perg, extracted, common_articles)
    results.append({
        'pathogen': pathogen,
        'model': model_name,
        'field': 'parameter_count',
        'precision': param_count_metrics['precision'],
        'recall': param_count_metrics['recall'],
        'f1': param_count_metrics['f1']
    })
    
    fields_to_evaluate = [
        ('unit', False, False),
        ('value', False, True),
        ('method', False, False),
        ('value_type', False, False),
        ('statistical_approach', False, False),
        ('paired_uncertainty', False, False),
        ('single_type_uncertainty', False, False),
        ('population_sex', False, False),
        ('population_group', False, False),
        ('population_sample_type', False, False),
    ]
    
    for field_name, is_multivalue, is_numeric in fields_to_evaluate:
        if field_name not in perg.columns or field_name not in extracted.columns:
            continue

        metrics = evaluate_parameter_field_optimal(perg, extracted, common_articles, field_name, is_multivalue, is_numeric)

        results.append({
            'pathogen': pathogen,
            'model': model_name,
            'field': field_name,
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1']
        })
    
    return results, filter_stats

## Parameter Extraction Results

In [44]:
FILTER_INVALID = False

all_results = []
all_filter_stats = []

for model_name in tqdm(models, desc="Evaluating models"):
    for pathogen in pathogens:
        try:
            results, filter_stats = evaluate_parameter_pathogen(
                pathogen, model_name, parameters_agentslr, perg_parameters_files, filter_invalid=FILTER_INVALID
            )
            all_results.extend(results)
            all_filter_stats.append(filter_stats)
        except Exception as e:
            print(f"Error evaluating {pathogen} with {model_name}: {e}")

results_df = pd.DataFrame(all_results)
filter_stats_df = pd.DataFrame(all_filter_stats)

Evaluating models: 100%|██████████| 5/5 [00:31<00:00,  6.29s/it]


In [45]:
for model_name in models:
    print(f"\n{'='*70}")
    print(f"Summary for Model: {model_name}")
    print(f"{'='*70}")
    
    model_df = results_df[results_df['model'] == model_name]
    
    pathogens_list = sorted(model_df['pathogen'].unique())
    
    summary_data = []
    for field in model_df['field'].unique():
        row = {'Field': PARAMETER_FIELD_DISPLAY_NAMES.get(field, field)}
        for pathogen in pathogens_list:
            field_data = model_df[(model_df['pathogen'] == pathogen) & (model_df['field'] == field)]
            if len(field_data) > 0:
                p = field_data.iloc[0]['precision']
                r = field_data.iloc[0]['recall']
                row[f"{pathogen}_P"] = p
                row[f"{pathogen}_R"] = r
                row[f"{pathogen}_F1"] = round(_f1_from_pr(p, r), 3)
            else:
                row[f"{pathogen}_P"] = np.nan
                row[f"{pathogen}_R"] = np.nan
                row[f"{pathogen}_F1"] = np.nan
        summary_data.append(row)
    
    summary = pd.DataFrame(summary_data)
    display(summary)


Summary for Model: oss


,Field,Ebola_P,Ebola_R,Ebola_F1,Lassa_P,Lassa_R,Lassa_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Parameter Counts,0.794,0.470,0.590,1.000,0.346,0.514,0.804,0.610,0.694,0.720,0.468,0.567
1,Unit,0.621,0.346,0.444,0.500,0.429,0.462,0.686,0.608,0.645,0.648,0.434,0.520
2,Value,0.204,0.204,0.204,0.222,0.222,0.222,0.234,0.234,0.234,0.138,0.138,0.138
3,Method,0.481,0.781,0.595,1.000,0.889,0.941,0.757,0.830,0.792,0.863,0.796,0.828
4,Value Type,0.302,0.333,0.317,0.375,0.429,0.400,0.346,0.569,0.430,0.123,0.222,0.158
5,Paired Uncertainty,0.592,0.725,0.652,0.250,0.400,0.308,0.391,0.878,0.541,0.461,0.898,0.609
6,Single Type Uncertainty,0.980,0.943,0.961,1.000,1.000,1.000,0.815,0.946,0.876,0.974,0.991,0.982
7,Population Sex,0.623,0.786,0.695,0.857,0.667,0.750,0.589,0.750,0.660,0.591,0.701,0.641
8,Population Group,0.245,0.325,0.279,0.143,0.111,0.125,0.226,0.250,0.237,0.544,0.539,0.541
9,Population Sample Type,0.321,0.405,0.358,0.857,0.667,0.750,0.581,0.629,0.604,0.368,0.365,0.366



Summary for Model: deepseek


,Field,Ebola_P,Ebola_R,Ebola_F1,Lassa_P,Lassa_R,Lassa_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Parameter Counts,0.571,0.588,0.579,0.464,0.575,0.514,0.596,0.603,0.599,0.501,0.599,0.546
1,Unit,0.853,0.376,0.522,0.923,0.316,0.471,0.897,0.578,0.703,0.620,0.310,0.413
2,Value,0.267,0.267,0.267,0.338,0.338,0.338,0.335,0.335,0.335,0.286,0.286,0.286
3,Method,0.775,0.821,0.797,0.727,0.716,0.721,0.847,0.887,0.867,0.910,0.794,0.848
4,Value Type,0.470,0.354,0.404,0.291,0.262,0.276,0.631,0.594,0.612,0.376,0.394,0.385
5,Paired Uncertainty,0.650,0.705,0.676,0.671,0.770,0.717,0.690,0.823,0.751,0.641,0.710,0.674
6,Single Type Uncertainty,0.638,0.953,0.764,0.587,0.936,0.722,0.636,0.904,0.747,0.582,1.000,0.736
7,Population Sex,0.532,0.323,0.402,0.619,0.338,0.437,0.422,0.270,0.329,0.486,0.343,0.402
8,Population Group,0.333,0.223,0.267,0.302,0.171,0.218,0.391,0.220,0.282,0.588,0.353,0.441
9,Population Sample Type,0.356,0.230,0.279,0.465,0.263,0.336,0.409,0.220,0.286,0.305,0.183,0.229



Summary for Model: kimi


,Field,Ebola_P,Ebola_R,Ebola_F1,Lassa_P,Lassa_R,Lassa_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Parameter Counts,0.524,0.730,0.610,0.593,0.812,0.685,0.452,0.724,0.557,0.549,0.733,0.628
1,Unit,0.839,0.525,0.646,0.915,0.505,0.651,0.701,0.653,0.676,0.560,0.286,0.379
2,Value,0.303,0.303,0.303,0.278,0.278,0.278,0.284,0.284,0.284,0.230,0.230,0.230
3,Method,0.653,0.831,0.731,0.631,0.765,0.692,0.712,0.903,0.796,0.809,0.740,0.773
4,Value Type,0.349,0.407,0.376,0.175,0.273,0.213,0.320,0.573,0.411,0.198,0.333,0.248
5,Paired Uncertainty,0.436,0.768,0.556,0.557,0.922,0.694,0.405,0.829,0.544,0.453,0.927,0.609
6,Single Type Uncertainty,0.959,0.931,0.945,0.907,0.990,0.947,0.850,0.961,0.902,0.958,0.996,0.977
7,Population Sex,0.720,0.685,0.702,0.851,0.796,0.823,0.624,0.768,0.689,0.514,0.573,0.542
8,Population Group,0.192,0.198,0.195,0.218,0.208,0.213,0.237,0.263,0.249,0.606,0.585,0.595
9,Population Sample Type,0.455,0.457,0.456,0.673,0.642,0.657,0.592,0.641,0.616,0.510,0.490,0.500



Summary for Model: glm


,Field,Ebola_P,Ebola_R,Ebola_F1,Lassa_P,Lassa_R,Lassa_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Parameter Counts,0.587,0.650,0.617,0.742,0.589,0.657,0.511,0.590,0.548,0.593,0.669,0.629
1,Unit,0.816,0.488,0.611,0.861,0.470,0.608,0.733,0.660,0.695,0.489,0.244,0.326
2,Value,0.204,0.204,0.204,0.182,0.182,0.182,0.169,0.169,0.169,0.160,0.160,0.160
3,Method,0.537,0.796,0.641,0.667,0.741,0.702,0.649,0.841,0.733,0.762,0.700,0.730
4,Value Type,0.306,0.394,0.344,0.145,0.205,0.170,0.304,0.530,0.386,0.200,0.330,0.249
5,Paired Uncertainty,0.453,0.837,0.588,0.459,0.718,0.560,0.513,0.881,0.648,0.521,0.934,0.669
6,Single Type Uncertainty,0.943,0.918,0.930,0.954,0.984,0.969,0.922,0.964,0.943,0.984,0.995,0.989
7,Population Sex,0.777,0.830,0.803,0.917,0.833,0.873,0.536,0.659,0.591,0.540,0.647,0.589
8,Population Group,0.137,0.160,0.148,0.233,0.215,0.224,0.214,0.242,0.227,0.594,0.581,0.587
9,Population Sample Type,0.385,0.441,0.411,0.767,0.708,0.736,0.571,0.621,0.595,0.465,0.453,0.459



Summary for Model: gpt


,Field,Ebola_P,Ebola_R,Ebola_F1,Lassa_P,Lassa_R,Lassa_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Parameter Counts,0.458,0.804,0.584,0.305,0.855,0.450,0.371,0.879,0.522,0.310,0.797,0.446
1,Unit,0.869,0.571,0.689,0.953,0.555,0.701,0.771,0.657,0.709,0.678,0.466,0.552
2,Value,0.305,0.305,0.305,0.304,0.304,0.304,0.284,0.284,0.284,0.248,0.248,0.248
3,Method,0.719,0.858,0.782,0.731,0.823,0.774,0.746,0.870,0.803,0.855,0.734,0.790
4,Value Type,0.387,0.431,0.408,0.139,0.211,0.168,0.363,0.550,0.437,0.251,0.414,0.313
5,Paired Uncertainty,0.487,0.772,0.597,0.654,0.861,0.743,0.524,0.894,0.661,0.498,0.944,0.652
6,Single Type Uncertainty,0.960,0.927,0.943,0.945,0.972,0.958,0.879,0.942,0.909,0.977,0.987,0.982
7,Population Sex,0.777,0.667,0.718,0.897,0.777,0.833,0.673,0.711,0.691,0.601,0.599,0.600
8,Population Group,0.207,0.191,0.199,0.649,0.562,0.602,0.137,0.133,0.135,0.491,0.444,0.466
9,Population Sample Type,0.497,0.455,0.475,0.660,0.571,0.612,0.635,0.606,0.620,0.422,0.381,0.400


## Parameter Flagging Evaluation

Evaluate parameter class flagging: did the AI correctly identify which articles contain each parameter class?

In [46]:
def compute_parameter_flagging_metrics(
    pathogen,
    model_name,
    perg_parameters_files,
    parameters_flagging_agentslr,
):
    df_perg_params = pd.read_csv(perg_parameters_files[pathogen])

    df_agentslr_params_flagging = pd.read_json(
        parameters_flagging_agentslr[(pathogen, model_name)], lines=True
    )

    df_perg_params["parameter_class"] = (
            df_perg_params["parameter_class"]
            .map(PARAMETER_CLASSES_MAPPING)
        )

    df_perg_params = (
        df_perg_params
        .groupby(["covidence_id", "parameter_class"])
        .agg("size")
        .reset_index()
        .drop(0, axis=1)
    )

    df_perg_params['article_id'] = df_perg_params['covidence_id'].astype(int)
    df_agentslr_params_flagging['article_id'] = df_agentslr_params_flagging['article_id'].astype(int)

    df_perg_params = df_perg_params[
            df_perg_params["article_id"].isin(df_agentslr_params_flagging["article_id"])
        ]

    df_perg_params = df_perg_params[
            df_perg_params["parameter_class"].isin(df_agentslr_params_flagging["parameter_class"])
        ]

    perg_positive = set(
        zip(df_perg_params["article_id"], df_perg_params["parameter_class"])
    )

    df_eval = df_agentslr_params_flagging.copy()

    df_eval["y_true"] = df_eval.apply(
        lambda r: int((r["article_id"], r["parameter_class"]) in perg_positive), axis=1
    )

    df_eval["y_pred"] = df_eval["contains_parameter"].astype(int)

    y_true = df_eval["y_true"]
    y_pred = df_eval["y_pred"]

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    return {
            "precision": round(float(precision), 3),
            "recall": round(float(recall), 3),
            "f1": round(float(f1), 3),
            "n": int(len(df_eval)),
        }

In [47]:
flagging_results = []

for model_name in models:
    print(f"\n{'='*50}")
    print(f"Flagging Evaluation - MODEL: {model_name}")
    print(f"{'='*50}")
    
    for pathogen in pathogens:
        print(f"Evaluating {pathogen}...")
        
        metrics = compute_parameter_flagging_metrics(
            pathogen,
            model_name,
            perg_parameters_files,
            parameters_flagging_agentslr,
        )
        
        flagging_results.append({
            'pathogen': pathogen,
            'model': model_name,
            'field': 'flagging',
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'n_articles': metrics['n'],
        })
        
        print(f"  P: {metrics['precision']:.3f}, R: {metrics['recall']:.3f}, F1: {metrics['f1']:.3f} (n={metrics['n']})")

flagging_df = pd.DataFrame(flagging_results)
print("\n" + "="*100)
print("FLAGGING RESULTS SUMMARY")
print("="*100)
display(flagging_df)


Flagging Evaluation - MODEL: oss
Evaluating Ebola...
  P: 0.596, R: 0.917, F1: 0.722 (n=1784)
Evaluating Lassa...
  P: 0.557, R: 0.982, F1: 0.711 (n=392)
Evaluating SARS...
  P: 0.500, R: 0.815, F1: 0.620 (n=824)
Evaluating Zika...
  P: 0.403, R: 0.957, F1: 0.567 (n=1328)

Flagging Evaluation - MODEL: deepseek
Evaluating Ebola...
  P: 0.492, R: 0.905, F1: 0.637 (n=1784)
Evaluating Lassa...
  P: 0.538, R: 0.909, F1: 0.676 (n=392)
Evaluating SARS...
  P: 0.391, R: 0.778, F1: 0.520 (n=824)
Evaluating Zika...
  P: 0.414, R: 0.937, F1: 0.575 (n=1288)

Flagging Evaluation - MODEL: kimi
Evaluating Ebola...
  P: 0.673, R: 0.896, F1: 0.769 (n=1784)
Evaluating Lassa...
  P: 0.703, R: 0.945, F1: 0.806 (n=392)
Evaluating SARS...
  P: 0.561, R: 0.685, F1: 0.617 (n=824)
Evaluating Zika...
  P: 0.561, R: 0.865, F1: 0.681 (n=1288)

Flagging Evaluation - MODEL: glm
Evaluating Ebola...
  P: 0.725, R: 0.822, F1: 0.770 (n=1784)
Evaluating Lassa...
  P: 0.774, R: 0.873, F1: 0.821 (n=392)
Evaluating SARS..

,pathogen,model,field,precision,recall,f1,n_articles
0,Ebola,oss,flagging,0.596,0.917,0.722,1784
1,Lassa,oss,flagging,0.557,0.982,0.711,392
2,SARS,oss,flagging,0.500,0.815,0.620,824
3,Zika,oss,flagging,0.403,0.957,0.567,1328
4,Ebola,deepseek,flagging,0.492,0.905,0.637,1784
5,Lassa,deepseek,flagging,0.538,0.909,0.676,392
6,SARS,deepseek,flagging,0.391,0.778,0.520,824
7,Zika,deepseek,flagging,0.414,0.937,0.575,1288
8,Ebola,kimi,flagging,0.673,0.896,0.769,1784
9,Lassa,kimi,flagging,0.703,0.945,0.806,392


## Save Parameter Extraction CSVs

In [48]:
df_params_all = results_df.copy()

for _, row in flagging_df.iterrows():
    df_params_all = pd.concat([df_params_all, pd.DataFrame([{
        'pathogen': row['pathogen'],
        'field': 'Article Flagging',
        'precision': row['precision'],
        'recall': row['recall'],
        'f1': row['f1'],
        'model': row['model']
    }])], ignore_index=True)

save_all_extraction_metrics(df_params_all, 'parameters', EVALS_PARAMETERS_DIR)
print("Parameter extraction metrics saved!")

Parameter extraction metrics saved!


---

# Latex Tables (Beautify the computed metrics.)

### Main-text Table for Data Extraction (Avg. across pathogens)
**Note:** <br>
* CSV paths `extraction_{data_tpe}_aggregated.csv` created in sections above (`notebooks/data_extraction_eval.ipynb`)
* Appendix tables for model ablations present in `notebooks/visualisations.ipynb`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PARAMS_PATH = Path("../data/agentslr/evals/data_extraction/parameters/extraction_parameters_aggregated.csv")
MODELS_PATH = Path("../data/agentslr/evals/data_extraction/models/extraction_models_aggregated.csv")
OUTBREAKS_PATH = Path("../data/agentslr/evals/data_extraction/outbreaks/extraction_outbreaks_aggregated.csv")

def _load_outbreaks_csv(outbreaks_path: Path) -> pd.DataFrame:
    if not outbreaks_path.exists():
        raise FileNotFoundError(f"No CSV found at {outbreaks_path}")
    return pd.read_csv(outbreaks_path)

def _to_long(df: pd.DataFrame, data_type: str) -> pd.DataFrame:
    d = df[df["model"].astype(str).str.lower().eq("oss")].copy()
    d = d.drop(columns=[c for c in ["overall_score"] if c in d.columns], errors="ignore")
    d["Data Type"] = data_type
    d = d.melt(
        id_vars=["Data Type", "pathogen", "metric"],
        value_vars=["flagging_score", "count_score", "extraction_score"],
        var_name="Stage",
        value_name="Score",
    )
    d["Stage"] = d["Stage"].map(
        {"flagging_score": "Flagging", "count_score": "Count", "extraction_score": "Extraction"}
    )
    d["metric"] = d["metric"].replace({"F1": "F1 Score"})
    return d[["Data Type", "Stage", "metric", "pathogen", "Score"]]

def build_data_extraction_table_df(
    params_path: Path = PARAMS_PATH,
    models_path: Path = MODELS_PATH,
    outbreaks_path: Path = OUTBREAKS_PATH,
    decimals: int = 2,
) -> pd.DataFrame:
    params = pd.read_csv(params_path)
    models = pd.read_csv(models_path)
    outbreaks = pd.read_csv(outbreaks_path)

    long = pd.concat(
        [
            _to_long(params, "Parameters"),
            _to_long(models, "Models"),
            _to_long(outbreaks, "Outbreaks"),
        ],
        ignore_index=True,
    )

    order_stages = ["Flagging", "Count", "Extraction"]
    order_metrics = ["Precision", "Recall", "F1 Score"]

    agg_dt = (
        long.groupby(["Data Type", "Stage", "metric"], as_index=False)["Score"]
        .agg(mean="mean", std=lambda s: float(np.std(s, ddof=0)))
    )

    avg_per_pathogen = (
        long.groupby(["pathogen", "Stage", "metric"], as_index=False)["Score"].mean()
        .assign(**{"Data Type": "Average (across data types)"})
    )
    agg_avg = (
        avg_per_pathogen.groupby(["Data Type", "Stage", "metric"], as_index=False)["Score"]
        .agg(mean="mean", std=lambda s: float(np.std(s, ddof=0)))
    )

    agg = pd.concat([agg_dt, agg_avg], ignore_index=True)

    fmt = lambda m, s: f"{m:.{decimals}f} ($\\pm$ {s:.{decimals}f})"
    agg["cell"] = [fmt(m, s) for m, s in zip(agg["mean"], agg["std"])]

    table = (
        agg.pivot(index=["Data Type", "Stage"], columns="metric", values="cell")
        .reindex(columns=order_metrics)
        .reindex(pd.MultiIndex.from_product(
            ["Parameters Models Outbreaks".split() + ["Average (across data types)"], order_stages]
        ), fill_value=np.nan)
    )
    table.index.names = ["Data Type", "Stage"]
    table.columns.name = None
    return table

In [3]:
table_df = build_data_extraction_table_df()

In [4]:
table_df

Precision             Recall  \
Data Type                   Stage                                              
Parameters                  Flagging    0.51 ($\pm$ 0.07)  0.92 ($\pm$ 0.06)   
                            Count       0.83 ($\pm$ 0.10)  0.47 ($\pm$ 0.09)   
                            Extraction  0.52 ($\pm$ 0.03)  0.57 ($\pm$ 0.04)   
Models                      Flagging    0.90 ($\pm$ 0.04)  0.91 ($\pm$ 0.05)   
                            Count       0.52 ($\pm$ 0.05)  0.99 ($\pm$ 0.01)   
                            Extraction  0.63 ($\pm$ 0.04)  0.74 ($\pm$ 0.02)   
Outbreaks                   Flagging    0.63 ($\pm$ 0.06)  0.76 ($\pm$ 0.05)   
                            Count       0.66 ($\pm$ 0.17)  0.72 ($\pm$ 0.28)   
                            Extraction  0.85 ($\pm$ 0.00)  0.76 ($\pm$ 0.02)   
Average (across data types) Flagging    0.70 ($\pm$ 0.05)  0.88 ($\pm$ 0.04)   
                            Count       0.67 ($\pm$ 0.09)  0.73 ($\pm$ 0.06)   
                            Extraction  0.62 ($\pm$ 0.07)  0.67 ($\pm$ 0.03)   

                                                 F1 Score  
Data Type                   Stage                          
Parameters                  Flagging    0.66 ($\pm$ 0.06)  
                            Count       0.59 ($\pm$ 0.07)  
                            Extraction  0.54 ($\pm$ 0.02)  
Models                      Flagging    0.91 ($\pm$ 0.04)  
                            Count       0.68 ($\pm$ 0.04)  
                            Extraction  0.67 ($\pm$ 0.03)  
Outbreaks                   Flagging    0.61 ($\pm$ 0.09)  
                            Count       0.69 ($\pm$ 0.22)  
                            Extraction  0.79 ($\pm$ 0.01)  
Average (across data types) Flagging    0.75 ($\pm$ 0.06)  
                            Count       0.65 ($\pm$ 0.06)  
                            Extraction  0.63 ($\pm$ 0.05)

In [ ]:

def df_to_latex_data_extraction_table(
    table_df: pd.DataFrame,
    caption: str,
    label: str,
) -> str:
    cols = ["Precision", "Recall", "F1 Score"]
    blocks = [
        ("Parameters", "Parameters"),
        ("Models", "Models"),
        ("Outbreaks", "Outbreaks"),
        ("Average (across data types)", "Average (across data types)"),
    ]
    stages = ["Flagging", "Count", "Extraction"]

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(r"\footnotesize")
    lines.append(r"\setlength{\tabcolsep}{4pt}")
    lines.append(r"\renewcommand{\arraystretch}{1.1}")
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Data Type} & $Precision$ & $Recall$ & $F_1 Score$ \\")
    lines.append(r"\midrule")
    for i, (key, title) in enumerate(blocks):
        if i > 0:
            lines.append("")
            lines.append(r"\midrule")
            lines.append("")
        lines.append(rf"\multicolumn{{4}}{{l}}{{\textbf{{{title}}}}} \\")
        for st in stages:
            row = table_df.loc[(key, st), cols].tolist()
            lines.append(rf"\quad {st:<10} & {row[0]} & {row[1]} & {row[2]} \\")
    lines.append("")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)

In [ ]:
latex = df_to_latex_data_extraction_table(
    table_df,
    caption=r"\textbf{Evaluation metrics (averaged across pathogens with $\pm$ deviation) for the data extraction stage of the \name{} pipeline (with \texttt{GPT-OSS-120B}).} Results are aggregated across pathogens and reported separately for \textit{Flagging}, \textit{Count}, and \textit{Extraction}. \textit{Flagging} measures identification of the presence of individual data components (epidemiological parameters, transmission models, and concluded outbreaks) in each article. \textit{Count} measures the accuracy of extracted quantities (number of components per article). \textit{Extraction} measures the accuracy of specific values extracted from articles, grouped by data component type.",
    label="tab:data_extraction_metrics_transposed",
)

In [12]:
print(latex)

\begin{table}[t]
\centering
\caption{\textbf{Evaluation metrics (averaged across pathogens with $\pm$ deviation) for the data extraction stage of the \name{} pipeline (with \texttt{GPT-OSS-120B}).} Results are aggregated across pathogens and reported separately for \textit{Flagging}, \textit{Count}, and \textit{Extraction}. \textit{Flagging} measures identification of the presence of individual data components (epidemiological parameters, transmission models, and concluded outbreaks) in each article. \textit{Count} measures the accuracy of extracted quantities (number of components per article). \textit{Extraction} measures the accuracy of specific values extracted from articles, grouped by data component type.}
\label{tab:data_extraction_metrics_transposed}
\footnotesize
\setlength{\tabcolsep}{4pt}
\renewcommand{\arraystretch}{1.1}
\begin{tabular}{lccc}
\toprule
\textbf{Data Type} & $Precision$ & $Recall$ & $F_1 Score$ \\
\midrule
\multicolumn{4}{l}{\textbf{Parameters}} \\
\quad Flagg

## Appendix Tables for Data Extraction — Parameters

In [5]:
import pandas as pd

CSV_PATH = '../data/agentslr/evals/data_extraction/parameters/extraction_parameters_detailed.csv'

PATHOGENS = ['Lassa', 'Ebola', 'SARS', 'Zika']

VALUE_FIELDS = ['value', 'unit', 'method']
UNCERTAINTY_FIELDS = ['value_type', 'statistical_approach', 'single_type_uncertainty', 'paired_uncertainty']
POPULATION_FIELDS = ['population_sex', 'population_group', 'population_sample_type']

FIELD_DISPLAY = {
    'value': 'value',
    'unit': 'unit',
    'method': 'method',
    'value_type': 'value type',
    'statistical_approach': 'statistical approach',
    'single_type_uncertainty': 'single type uncertainty',
    'paired_uncertainty': 'paired uncertainty',
    'population_sex': 'population sex',
    'population_group': 'population group',
    'population_sample_type': 'population sample type',
}

def _fmt(val: float) -> str:
    if pd.isna(val):
        return '--'
    return f'{val:.2f}'

In [6]:
def build_flagging_count_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    oss = df[df['model'] == 'oss'].copy()

    rows = []
    for metric, field in [('Flagging', 'Article Flagging'), ('Count', 'parameter_count')]:
        row = {'Metric': metric}
        for p in PATHOGENS:
            sub = oss[(oss['pathogen'] == p) & (oss['field'] == field)]
            if sub.empty:
                row[f'{p}_P'] = float('nan')
                row[f'{p}_R'] = float('nan')
                row[f'{p}_F1'] = float('nan')
            else:
                row[f'{p}_P'] = sub['precision'].values[0]
                row[f'{p}_R'] = sub['recall'].values[0]
                row[f'{p}_F1'] = sub['f1'].values[0]
        rows.append(row)

    return pd.DataFrame(rows).set_index('Metric')

In [7]:
df1 = build_flagging_count_df(CSV_PATH)
df1

,Lassa_P,Lassa_R,Lassa_F1,Ebola_P,Ebola_R,Ebola_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
Metric,,,,,,,,,,,,
Flagging,0.557,0.982,0.711,0.596,0.917,0.722,0.500,0.815,0.620,0.403,0.957,0.567
Count,1.000,0.346,0.514,0.794,0.470,0.590,0.804,0.610,0.694,0.720,0.468,0.567


In [8]:
def flagging_count_to_latex(df: pd.DataFrame) -> str:
    lines = []
    lines.append(r'\begin{table}[h]')
    lines.append(r'\footnotesize')
    lines.append(r'\centering')
    lines.append(r'\caption{\textbf{Flagging and Count classification metrics for parameter extraction with \texttt{GPT-OSS-120B}.} $P=\text{precision}$; $R=\text{recall}$; $F_1=\text{F1-Score}$.}')
    lines.append(r'\label{tab:parameters_extended_flagging_count}')
    lines.append(r'\begin{tabular}{l|cccccccccccc}')
    lines.append(r'\toprule')
    lines.append(
        r'\bf Metric & \multicolumn{3}{c}{\textbf{Lassa}} '
        r'& \multicolumn{3}{c}{\textbf{Ebola}} '
        r'& \multicolumn{3}{c}{\textbf{SARS}} '
        r'& \multicolumn{3}{c}{\textbf{Zika}} \\'
    )
    lines.append(
        r' & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$ \\'
    )
    lines.append(r'\midrule')

    for i, metric in enumerate(df.index):
        row = df.loc[metric]
        cells = []
        for p in PATHOGENS:
            cells.append(_fmt(row[f'{p}_P']))
            cells.append(_fmt(row[f'{p}_R']))
            cells.append(_fmt(row[f'{p}_F1']))
        lines.append(r'\bf ' + metric + ' & ' + ' &\n'.join(cells) + r' \\')
        if i < len(df.index) - 1:
            lines.append(r'\midrule')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)


In [9]:
latex1 = flagging_count_to_latex(df1)
print(latex1)

\begin{table}[h]
\footnotesize
\centering
\caption{\textbf{Flagging and Count classification metrics for parameter extraction with \texttt{GPT-OSS-120B}.} $P=\text{precision}$; $R=\text{recall}$; $F_1=\text{F1-Score}$.}
\label{tab:parameters_extended_flagging_count}
\begin{tabular}{l|cccccccccccc}
\toprule
\bf Metric & \multicolumn{3}{c}{\textbf{Lassa}} & \multicolumn{3}{c}{\textbf{Ebola}} & \multicolumn{3}{c}{\textbf{SARS}} & \multicolumn{3}{c}{\textbf{Zika}} \\
 & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ \\
\midrule
\bf Flagging & 0.56 &
0.98 &
0.71 &
0.60 &
0.92 &
0.72 &
0.50 &
0.81 &
0.62 &
0.40 &
0.96 &
0.57 \\
\midrule
\bf Count & 1.00 &
0.35 &
0.51 &
0.79 &
0.47 &
0.59 &
0.80 &
0.61 &
0.69 &
0.72 &
0.47 &
0.57 \\
\bottomrule
\end{tabular}
\end{table}


In [10]:
def build_field_level_df(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    oss = df[df['model'] == 'oss'].copy()

    all_field_groups = [
        ('Value', VALUE_FIELDS),
        ('Uncertainty', UNCERTAINTY_FIELDS),
        ('Population', POPULATION_FIELDS),
    ]

    rows = []

    for group_name, fields in all_field_groups:
        group_rows = []
        for field in fields:
            row = {'Group': group_name, 'Field': FIELD_DISPLAY[field]}
            for p in PATHOGENS:
                sub = oss[(oss['pathogen'] == p) & (oss['field'] == field)]
                if sub.empty:
                    row[f'{p}_P'] = float('nan')
                    row[f'{p}_R'] = float('nan')
                    row[f'{p}_F1'] = float('nan')
                else:
                    row[f'{p}_P'] = sub['precision'].values[0]
                    row[f'{p}_R'] = sub['recall'].values[0]
                    row[f'{p}_F1'] = sub['f1'].values[0]
            group_rows.append(row)
            rows.append(row)

        avg_row = {'Group': group_name, 'Field': 'Average'}
        for p in PATHOGENS:
            p_vals = [r[f'{p}_P'] for r in group_rows if not pd.isna(r[f'{p}_P'])]
            r_vals = [r[f'{p}_R'] for r in group_rows if not pd.isna(r[f'{p}_R'])]
            f_vals = [r[f'{p}_F1'] for r in group_rows if not pd.isna(r[f'{p}_F1'])]
            avg_row[f'{p}_P'] = sum(p_vals) / len(p_vals) if p_vals else float('nan')
            avg_row[f'{p}_R'] = sum(r_vals) / len(r_vals) if r_vals else float('nan')
            avg_row[f'{p}_F1'] = sum(f_vals) / len(f_vals) if f_vals else float('nan')
        rows.append(avg_row)

    overall_row = {'Group': 'Overall', 'Field': ''}
    all_fields = VALUE_FIELDS + UNCERTAINTY_FIELDS + POPULATION_FIELDS
    for p in PATHOGENS:
        p_vals, r_vals, f_vals = [], [], []
        for field in all_fields:
            sub = oss[(oss['pathogen'] == p) & (oss['field'] == field)]
            if not sub.empty:
                p_vals.append(sub['precision'].values[0])
                r_vals.append(sub['recall'].values[0])
                f_vals.append(sub['f1'].values[0])
        overall_row[f'{p}_P'] = sum(p_vals) / len(p_vals) if p_vals else float('nan')
        overall_row[f'{p}_R'] = sum(r_vals) / len(r_vals) if r_vals else float('nan')
        overall_row[f'{p}_F1'] = sum(f_vals) / len(f_vals) if f_vals else float('nan')
    rows.append(overall_row)

    return pd.DataFrame(rows)



In [11]:
df2 = build_field_level_df(CSV_PATH)

In [12]:
df2

,Group,Field,Lassa_P,Lassa_R,Lassa_F1,Ebola_P,Ebola_R,Ebola_F1,SARS_P,SARS_R,SARS_F1,Zika_P,Zika_R,Zika_F1
0,Value,value,0.222000,0.222000,0.222000,0.204000,0.204000,0.204000,0.234000,0.234000,0.234000,0.138000,0.13800,0.138000
1,Value,unit,0.500000,0.429000,0.462000,0.621000,0.346000,0.444000,0.686000,0.608000,0.645000,0.648000,0.43400,0.520000
2,Value,method,1.000000,0.889000,0.941000,0.481000,0.781000,0.595000,0.757000,0.830000,0.792000,0.863000,0.79600,0.828000
3,Value,Average,0.574000,0.513333,0.541667,0.435333,0.443667,0.414333,0.559000,0.557333,0.557000,0.549667,0.45600,0.495333
4,Uncertainty,value type,0.375000,0.429000,0.400000,0.302000,0.333000,0.317000,0.346000,0.569000,0.430000,0.123000,0.22200,0.158000
5,Uncertainty,statistical approach,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.439000,0.65800,0.526000
6,Uncertainty,single type uncertainty,1.000000,1.000000,1.000000,0.980000,0.943000,0.962000,0.815000,0.946000,0.876000,0.974000,0.99100,0.982000
7,Uncertainty,paired uncertainty,0.250000,0.400000,0.308000,0.592000,0.725000,0.652000,0.391000,0.878000,0.541000,0.461000,0.89800,0.609000
8,Uncertainty,Average,0.541667,0.609667,0.569333,0.624667,0.667000,0.643667,0.517333,0.797667,0.615667,0.499250,0.69225,0.568750
9,Population,population sex,0.857000,0.667000,0.750000,0.623000,0.786000,0.695000,0.589000,0.750000,0.660000,0.591000,0.70100,0.642000


In [13]:
def field_level_to_latex(df: pd.DataFrame) -> str:
    lines = []
    lines.append(r'\begin{table}[h]')
    lines.append(r'\setlength{\tabcolsep}{3pt}')
    lines.append(r'\footnotesize')
    lines.append(r'\centering')
    lines.append(r'\caption{\textbf{Field-level precision, recall, and F1 for Extraction on parameters with \texttt{GPT-OSS-120B}.} \textit{Group} corresponds to the sub-stage of parameter extraction where the field is collected. The final row shows averages across all fields.}')
    lines.append(r'\label{tab:parameters_detailed_metrics}')
    lines.append(r'\begin{tabular}{ll|ccc|ccc|ccc|ccc}')
    lines.append(r'\toprule')
    lines.append(
        r'\bf Group & \bf Field & \multicolumn{3}{c}{\textbf{Lassa}} '
        r'& \multicolumn{3}{c}{\textbf{Ebola}} '
        r'& \multicolumn{3}{c}{\textbf{SARS}} '
        r'& \multicolumn{3}{c}{\textbf{Zika}} \\'
    )
    lines.append(
        r' & & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$'
        r' & $P$ & $R$ & $F_1$ \\'
    )
    lines.append(r'\midrule')

    group_order = ['Value', 'Uncertainty', 'Population']
    group_sizes = {
        'Value': len(VALUE_FIELDS) + 1,
        'Uncertainty': len(UNCERTAINTY_FIELDS) + 1,
        'Population': len(POPULATION_FIELDS) + 1,
    }

    for g_idx, group in enumerate(group_order):
        group_df = df[df['Group'] == group].reset_index(drop=True)
        n = group_sizes[group]

        for i, row in group_df.iterrows():
            cells = ' & '.join(
                _fmt(row[f'{p}_{m}']) for p in PATHOGENS for m in ['P', 'R', 'F1']
            )
            group_col = r'\multirow{' + str(n) + r'}{*}{\textbf{' + group + r'}}' if i == 0 else ''
            field_display = row['Field']

            if field_display == 'Average':
                lines.append(r'\cmidrule{2-14}')

            lines.append(f'{group_col} & {field_display} & {cells} \\\\')

        if g_idx < len(group_order) - 1:
            lines.append(r'\midrule')

    lines.append(r'\midrule')

    overall = df[df['Group'] == 'Overall'].iloc[0]
    cells = ' & '.join(
        _fmt(overall[f'{p}_{m}']) for p in PATHOGENS for m in ['P', 'R', 'F1']
    )
    lines.append(r'\textbf{Overall} & & ' + cells + r' \\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')
    return '\n'.join(lines)

In [14]:
latex2 = field_level_to_latex(df2)
print(latex2)

\begin{table}[h]
\setlength{\tabcolsep}{3pt}
\footnotesize
\centering
\caption{\textbf{Field-level precision, recall, and F1 for Extraction on parameters with \texttt{GPT-OSS-120B}.} \textit{Group} corresponds to the sub-stage of parameter extraction where the field is collected. The final row shows averages across all fields.}
\label{tab:parameters_detailed_metrics}
\begin{tabular}{ll|ccc|ccc|ccc|ccc}
\toprule
\bf Group & \bf Field & \multicolumn{3}{c}{\textbf{Lassa}} & \multicolumn{3}{c}{\textbf{Ebola}} & \multicolumn{3}{c}{\textbf{SARS}} & \multicolumn{3}{c}{\textbf{Zika}} \\
 & & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ & $P$ & $R$ & $F_1$ \\
\midrule
\multirow{4}{*}{\textbf{Value}} & value & 0.22 & 0.22 & 0.22 & 0.20 & 0.20 & 0.20 & 0.23 & 0.23 & 0.23 & 0.14 & 0.14 & 0.14 \\
 & unit & 0.50 & 0.43 & 0.46 & 0.62 & 0.35 & 0.44 & 0.69 & 0.61 & 0.65 & 0.65 & 0.43 & 0.52 \\
 & method & 1.00 & 0.89 & 0.94 & 0.48 & 0.78 & 0.59 & 0.76 & 0.83 & 0.79 & 0.86 & 0.80 & 0.83 \\


## Appendix Tables for Data Extraction — Models

In [15]:
import pandas as pd

CSV_PATH = '../data/agentslr/evals/data_extraction/models/extraction_models_detailed.csv'

FIELD_ORDER = [
    ('Article Flagging',            'Flagging',    'Article Flagging'),
    ('model_count',                 'Counts',      'Model Count'),
    ('model_type',                  'Extraction',  'Model Type'),
    ('compartmental_type',          'Extraction',  'Compartmental Type'),
    ('stoch_deter',                 'Extraction',  'Stochastic vs Deterministic'),
    ('theoretical_model',           'Extraction',  'Theoretical vs Data-Fitted'),
    ('code_available',              'Extraction',  'Code Available'),
    ('transmission_route',          'Extraction',  'Transmission Routes'),
    ('assumptions',                 'Extraction',  'Assumptions'),
    ('interventions_type',          'Extraction',  'Interventions'),
]

PATHOGENS = ['Lassa', 'Ebola', 'SARS', 'Zika']

EBOLA_NO_COMPARTMENTAL = True

In [16]:

def load_and_transform(csv_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(csv_path)
    df = df[df['model'] == 'oss'].copy()

    pivot = df.pivot_table(
        index=['pathogen', 'field'],
        values=['precision', 'recall', 'f1'],
        aggfunc='first'
    ).reset_index()

    wide = pivot.pivot(index='field', columns='pathogen', values=['precision', 'recall', 'f1'])
    wide.columns = [f'{pathogen}_{metric}' for metric, pathogen in wide.columns]
    wide = wide.reset_index()

    overall = df.groupby('pathogen')[['precision', 'recall', 'f1']].mean().reset_index()
    overall_wide = overall.set_index('pathogen').stack().reset_index()
    overall_wide.columns = ['pathogen', 'metric', 'value']
    overall_wide['col'] = overall_wide['pathogen'] + '_' + overall_wide['metric']
    overall_row = overall_wide.pivot_table(index=None, columns='col', values='value', aggfunc='first')
    overall_row.insert(0, 'field', 'overall')

    return wide, overall_row


In [17]:
wide, overall_row = load_and_transform(CSV_PATH)

In [18]:
wide

,field,Ebola_precision,Lassa_precision,SARS_precision,Zika_precision,Ebola_recall,Lassa_recall,SARS_recall,Zika_recall,Ebola_f1,Lassa_f1,SARS_f1,Zika_f1
0,Article Flagging,0.915,0.955,0.861,0.868,0.917,0.987,0.861,0.892,0.915,0.970,0.861,0.876
1,assumptions,0.270,0.294,0.215,0.308,0.462,0.455,0.389,0.522,0.341,0.357,0.277,0.387
2,code_available,0.850,1.000,1.000,0.821,0.840,0.889,1.000,0.762,0.845,0.941,1.000,0.790
3,compartmental_type,NaN,0.000,0.800,0.828,NaN,0.000,0.800,0.828,NaN,0.000,0.800,0.828
4,interventions_type,0.478,0.538,0.464,0.316,0.688,0.636,0.788,0.692,0.564,0.583,0.584,0.434
5,model_count,0.500,0.600,0.486,0.483,1.000,1.000,0.971,0.977,0.667,0.750,0.648,0.646
6,model_type,0.889,0.889,0.765,0.881,0.889,0.889,0.765,0.881,0.889,0.889,0.765,0.881
7,screening,0.500,0.600,0.486,0.483,1.000,1.000,0.971,0.977,0.667,0.750,0.648,0.646
8,stoch_deter,0.753,1.000,0.758,0.825,0.853,1.000,0.781,0.786,0.800,1.000,0.769,0.805
9,theoretical_model,0.877,0.778,0.618,0.810,0.877,0.778,0.618,0.810,0.877,0.778,0.618,0.810


In [19]:
overall_row

col,field,Ebola_f1,Ebola_precision,Ebola_recall,Lassa_f1,Lassa_precision,Lassa_recall,SARS_f1,SARS_precision,SARS_recall,Zika_f1,Zika_precision,Zika_recall
value,overall,0.6702,0.6158,0.7676,0.726273,0.695818,0.779818,0.659909,0.610455,0.751636,0.710273,0.663818,0.806545


In [20]:
def build_latex_table(wide: pd.DataFrame, overall_row: pd.DataFrame) -> str:
    def fmt(val):
        return f'{val:.2f}'

    def get_cell(field, pathogen):
        if field == 'compartmental_type' and pathogen == 'Ebola':
            return '---', '---', '---'
        row = wide[wide['field'] == field]
        if row.empty:
            return '---', '---', '---'
        p = fmt(row[f'{pathogen}_precision'].values[0])
        r = fmt(row[f'{pathogen}_recall'].values[0])
        f = fmt(row[f'{pathogen}_f1'].values[0])
        return p, r, f

    def get_overall(pathogen):
        p = fmt(overall_row[f'{pathogen}_precision'].values[0])
        r = fmt(overall_row[f'{pathogen}_recall'].values[0])
        f = fmt(overall_row[f'{pathogen}_f1'].values[0])
        return p, r, f

    lines = []
    lines.append(r'\begin{table}[h]')
    lines.append(r'\centering')
    lines.append(r'\caption{\textbf{Precision, recall, and F1 metrics for transmission model screening and extraction across four pathogens.} \textit{Screening} includes article flagging and model count accuracy. \textit{Extraction} evaluates field-level accuracy for matched model pairs, covering core structural characteristics (model type, stochastic vs deterministic, theoretical vs data-fitted, code availability) and more complex multi-value fields (transmission routes, assumptions, interventions). Strong performance is observed for core model characteristics, while extraction of assumptions, interventions, and transmission routes remains more challenging.}')
    lines.append(r'\label{tab:comprehensive_metrics}')
    lines.append(r'\footnotesize')
    lines.append(r'\begin{tabular}{l|cccccccccccc}')
    lines.append(r'\toprule')

    header1 = r' & ' + ' & '.join(
        rf'\multicolumn{{3}}{{c}}{{\textbf{{{p}}}}}' for p in PATHOGENS
    ) + r' \\'
    lines.append(header1)

    header2 = r' & ' + ' & '.join([r'$P$ & $R$ & $F1$'] * len(PATHOGENS)) + r' \\'
    lines.append(header2)
    lines.append(r'\midrule')

    current_section = None
    for field_key, section, display_name in FIELD_ORDER:
        if section != current_section:
            current_section = section
            lines.append(rf'\multicolumn{{13}}{{l}}{{\textbf{{{section}}}}} \\')

        cells = []
        for pathogen in PATHOGENS:
            p, r, f = get_cell(field_key, pathogen)
            cells.append(f'{p} & {r} & {f}')

        lines.append(display_name)
        lines.append('& ' + '\n& '.join(cells) + r' \\')
        lines.append(r'\midrule')

    overall_cells = []
    for pathogen in PATHOGENS:
        p, r, f = get_overall(pathogen)
        overall_cells.append(f'{p} & {r} & {f}')

    lines.append(r'\textbf{Overall}')
    lines.append('& ' + '\n& '.join(overall_cells) + r' \\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table}')

    return '\n'.join(lines)

In [21]:
latex = build_latex_table(wide, overall_row)
print(latex)

\begin{table}[h]
\centering
\caption{\textbf{Precision, recall, and F1 metrics for transmission model screening and extraction across four pathogens.} \textit{Screening} includes article flagging and model count accuracy. \textit{Extraction} evaluates field-level accuracy for matched model pairs, covering core structural characteristics (model type, stochastic vs deterministic, theoretical vs data-fitted, code availability) and more complex multi-value fields (transmission routes, assumptions, interventions). Strong performance is observed for core model characteristics, while extraction of assumptions, interventions, and transmission routes remains more challenging.}
\label{tab:comprehensive_metrics}
\footnotesize
\begin{tabular}{l|cccccccccccc}
\toprule
 & \multicolumn{3}{c}{\textbf{Lassa}} & \multicolumn{3}{c}{\textbf{Ebola}} & \multicolumn{3}{c}{\textbf{SARS}} & \multicolumn{3}{c}{\textbf{Zika}} \\
 & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ & $P$ & $R$ & $F1$ \\
\mid

## Appendix Tables for Data Extraction — Outbreaks

In [22]:
import pandas as pd
import numpy as np

CSV_PATH_1 = '../data/agentslr/evals/data_extraction/outbreaks/extraction_outbreaks_detailed.csv'
CSV_PATH_2 = '../data/agentslr/evals/data_extraction/outbreaks/all_outbreak_extractions_results.csv'

TEMPORAL_FIELDS = [
    'outbreak_start_year', 'outbreak_start_month', 'outbreak_start_day',
    'outbreak_end_month', 'outbreak_end_day', 'outbreak_duration_months',
]
GEOGRAPHIC_FIELDS = [
    'outbreak_country', 'outbreak_location',
]
CASE_BURDEN_FIELDS = [
    'cases_confirmed', 'cases_suspected', 'cases_asymptomatic', 'cases_severe', 'deaths',
]
EPIDEMIOLOGICAL_FIELDS = [
    'mode_of_detection', 'pre_outbreak',
    'outbreak_is_currently_ongoing', 'asymptomatic_transmission_described',
]

CATEGORIES = {
    'Temporal Features': TEMPORAL_FIELDS,
    'Geographic and Spatial Features': GEOGRAPHIC_FIELDS,
    'Case Burden': CASE_BURDEN_FIELDS,
    'Epidemiological Context and Metadata': EPIDEMIOLOGICAL_FIELDS,
}

FIELD_DISPLAY = {
    'outbreak_start_year':              'Start Year',
    'outbreak_start_month':             'Start Month',
    'outbreak_start_day':               'Start Day',
    'outbreak_end_month':               'End Month',
    'outbreak_end_day':                 'End Day',
    'outbreak_duration_months':         'Duration (Months)',
    'outbreak_country':                 'Outbreak Country',
    'outbreak_location':                'Location',
    'cases_confirmed':                  'Confirmed Cases',
    'cases_suspected':                  'Suspected Cases',
    'cases_asymptomatic':               'Asymptomatic Cases',
    'cases_severe':                     'Severe Cases',
    'deaths':                           'Deaths',
    'mode_of_detection':                'Mode of Detection',
    'pre_outbreak':                     'Pre-outbreak Status',
    'outbreak_is_currently_ongoing':    'Ongoing Status',
    'asymptomatic_transmission_described': 'Asymptomatic Transmission',
}

ALL_EXTRACTION_FIELDS = [f for fields in CATEGORIES.values() for f in fields]

In [23]:
ALL_EXTRACTION_FIELDS

['outbreak_start_year',
 'outbreak_start_month',
 'outbreak_start_day',
 'outbreak_end_month',
 'outbreak_end_day',
 'outbreak_duration_months',
 'outbreak_country',
 'outbreak_location',
 'cases_confirmed',
 'cases_suspected',
 'cases_asymptomatic',
 'cases_severe',
 'deaths',
 'mode_of_detection',
 'pre_outbreak',
 'outbreak_is_currently_ongoing',
 'asymptomatic_transmission_described']

In [24]:
def _f1(p, r):
    denom = p + r
    return np.where(denom > 0, 2 * p * r / denom, 0.0)


def _pivot_pathogen(df, pathogen):
    return df[df['pathogen'] == pathogen].set_index('field')


def _category_means(sub, fields):
    available = [f for f in fields if f in sub.index]
    return sub.loc[available, ['precision', 'recall']].mean() if available else pd.Series({'precision': np.nan, 'recall': np.nan})


def _build_row(metric, field, lp, lr, zp, zr):
    lf = float(_f1(lp, lr))
    zf = float(_f1(zp, zr))
    return {
        'metric': metric, 'field': field,
        'lassa_p': lp, 'lassa_r': lr, 'lassa_f1': lf,
        'zika_p':  zp, 'zika_r':  zr, 'zika_f1':  zf,
    }

In [25]:
def transform_aggregated(csv_path):
    raw = pd.read_csv(csv_path)
    raw = raw[raw['model'] == 'oss']

    lassa = _pivot_pathogen(raw, 'Lassa')
    zika  = _pivot_pathogen(raw, 'Zika')

    rows = []

    rows.append(_build_row(
        'Flagging', 'Article Flagging',
        lassa.loc['Article Flagging', 'precision'], lassa.loc['Article Flagging', 'recall'],
        zika.loc['Article Flagging',  'precision'], zika.loc['Article Flagging',  'recall'],
    ))

    rows.append(_build_row(
        'Counts', 'Outbreak Counts',
        lassa.loc['outbreak_count', 'precision'], lassa.loc['outbreak_count', 'recall'],
        zika.loc['outbreak_count',  'precision'], zika.loc['outbreak_count',  'recall'],
    ))

    for cat_name, cat_fields in CATEGORIES.items():
        lm = _category_means(lassa, cat_fields)
        zm = _category_means(zika,  cat_fields)
        rows.append(_build_row('Extraction', cat_name, lm['precision'], lm['recall'], zm['precision'], zm['recall']))

    lall = _category_means(lassa, ALL_EXTRACTION_FIELDS)
    zall = _category_means(zika,  ALL_EXTRACTION_FIELDS)
    rows.append(_build_row('Overall', 'Overall', lall['precision'], lall['recall'], zall['precision'], zall['recall']))

    return pd.DataFrame(rows)


In [26]:
agg_df = transform_aggregated(CSV_PATH_2)
agg_df

,metric,field,lassa_p,lassa_r,lassa_f1,zika_p,zika_r,zika_f1
0,Flagging,Article Flagging,0.690000,0.816000,0.747729,0.576000,0.713000,0.637220
1,Counts,Outbreak Counts,0.833000,1.000000,0.908893,0.489000,0.449000,0.468147
2,Extraction,Temporal Features,0.831800,0.738000,0.782098,0.849800,0.815400,0.832245
3,Extraction,Geographic and Spatial Features,0.750000,0.778000,0.763743,0.750000,0.750000,0.750000
4,Extraction,Case Burden,0.825000,0.750000,0.785714,0.928500,0.928500,0.928500
5,Extraction,Epidemiological Context and Metadata,0.928500,0.700000,0.798219,0.841000,0.670500,0.746134
6,Overall,Overall,0.847923,0.734308,0.787036,0.843846,0.778154,0.809670


In [27]:
def transform_detailed(csv_path):
    raw = pd.read_csv(csv_path)
    raw = raw[raw['model'] == 'oss']

    lassa = _pivot_pathogen(raw, 'Lassa')
    zika  = _pivot_pathogen(raw, 'Zika')

    rows = []

    for cat_name, cat_fields in CATEGORIES.items():
        for field in cat_fields:
            if field not in lassa.index and field not in zika.index:
                continue
            lp = lassa.loc[field, 'precision'] if field in lassa.index else np.nan
            lr = lassa.loc[field, 'recall']    if field in lassa.index else np.nan
            zp = zika.loc[field,  'precision'] if field in zika.index  else np.nan
            zr = zika.loc[field,  'recall']    if field in zika.index  else np.nan
            rows.append(_build_row(cat_name, FIELD_DISPLAY.get(field, field), lp, lr, zp, zr))

    lall = _category_means(lassa, ALL_EXTRACTION_FIELDS)
    zall = _category_means(zika,  ALL_EXTRACTION_FIELDS)
    rows.append(_build_row('Overall', 'Overall', lall['precision'], lall['recall'], zall['precision'], zall['recall']))

    return pd.DataFrame(rows)



In [28]:
det_df = transform_detailed(CSV_PATH_1)
det_df

,metric,field,lassa_p,lassa_r,lassa_f1,zika_p,zika_r,zika_f1
0,Temporal Features,Start Year,0.889000,0.800000,0.842155,0.900000,0.818000,0.857043
1,Temporal Features,Start Month,0.778000,0.778000,0.778000,0.800000,0.800000,0.800000
2,Temporal Features,Start Day,0.857000,0.667000,0.750156,0.952000,0.952000,0.952000
3,Temporal Features,End Month,0.778000,0.778000,0.778000,0.650000,0.650000,0.650000
4,Temporal Features,End Day,0.857000,0.667000,0.750156,0.947000,0.857000,0.899755
5,Geographic and Spatial Features,Outbreak Country,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
6,Geographic and Spatial Features,Location,0.500000,0.556000,0.526515,0.500000,0.500000,0.500000
7,Case Burden,Confirmed Cases,0.750000,0.600000,0.666667,0.857000,0.857000,0.857000
8,Case Burden,Deaths,0.900000,0.900000,0.900000,1.000000,1.000000,1.000000
9,Epidemiological Context and Metadata,Mode of Detection,0.714000,0.500000,0.588138,0.500000,0.409000,0.449945


In [29]:

def _fmt(x):
    return f'{x:.2f}'


def _row_str(cells):
    return ' & '.join(cells) + r' \\'



In [30]:

def to_latex_aggregated(df):
    lines = [
        r'\begin{table}[h]',
        r'\centering',
        r'\caption{\textbf{Precision, recall, and F1 for automated outbreak screening and extraction across major feature categories, evaluated against expert-curated PERG database}. \textit{Screening} measured article flagging (identifying papers containing outbreaks) and outbreak count accuracy (extracting the correct number of outbreaks per paper). \textit{Extraction} evaluated field-level accuracy for matched outbreak pairs across four epidemiological categories: temporal features (start/end dates, duration), geographic features (country, specific location), case burden (confirmed/suspected/asymptomatic/severe cases, deaths), and epidemiological context (detection mode, pre-outbreak status, asymptomatic transmission). Overall metrics represent the average across all extraction fields.}',
        r'\footnotesize',
        r'\label{tab:outbreak_aggregated_metrics}',
        r'\begin{tabular}{ll|ccc|ccc}',
        r'\toprule',
        r' \multirow{2}{*}{\bf Metric} & \multirow{2}{*}{\bf Field}',
        r' & \multicolumn{3}{c}{\textbf{Lassa}}',
        r' & \multicolumn{3}{c}{\textbf{Zika}} \\',
        r' &  & Precision & Recall & F1 & Precision & Recall & F1 \\',
        r'\midrule',
    ]

    def data_cols(r):
        return [_fmt(r.lassa_p), _fmt(r.lassa_r), _fmt(r.lassa_f1),
                _fmt(r.zika_p),  _fmt(r.zika_r),  _fmt(r.zika_f1)]

    for metric, label in [('Flagging', r'\bf Flagging'), ('Counts', r'\bf Counts')]:
        r = df[df['metric'] == metric].iloc[0]
        lines.append(_row_str([label, r['field']] + data_cols(r)))
        lines.append(r'\midrule')

    ext = df[df['metric'] == 'Extraction'].reset_index(drop=True)
    n   = len(ext)
    for i, (_, r) in enumerate(ext.iterrows()):
        prefix = rf'\multirow{{{n}}}{{*}}{{\textbf{{Extraction}}}}' if i == 0 else ''
        lines.append(_row_str([prefix, r['field']] + data_cols(r)))
    lines.append(r'\midrule')

    r = df[df['metric'] == 'Overall'].iloc[0]
    lines.append(_row_str([r'\textbf{Overall}', ''] + data_cols(r)))

    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)


In [31]:
print(to_latex_aggregated(agg_df))

\begin{table}[h]
\centering
\caption{\textbf{Precision, recall, and F1 for automated outbreak screening and extraction across major feature categories, evaluated against expert-curated PERG database}. \textit{Screening} measured article flagging (identifying papers containing outbreaks) and outbreak count accuracy (extracting the correct number of outbreaks per paper). \textit{Extraction} evaluated field-level accuracy for matched outbreak pairs across four epidemiological categories: temporal features (start/end dates, duration), geographic features (country, specific location), case burden (confirmed/suspected/asymptomatic/severe cases, deaths), and epidemiological context (detection mode, pre-outbreak status, asymptomatic transmission). Overall metrics represent the average across all extraction fields.}
\footnotesize
\label{tab:outbreak_aggregated_metrics}
\begin{tabular}{ll|ccc|ccc}
\toprule
 \multirow{2}{*}{\bf Metric} & \multirow{2}{*}{\bf Field}
 & \multicolumn{3}{c}{\textbf{La

In [32]:
def to_latex_detailed(df):
    lines = [
        r'\begin{table}[h]',
        r'\centering',
        r'\caption{\textbf{Detailed precision, recall, and F1 for outbreak feature extraction by category.} Each row shows field-level performance within the four major epidemiological categories. Temporal and case burden features showed consistently high performance, while location-specific fields and epidemiological context features showed greater variability. Overall metrics represent the average across all 17 extraction fields.}',
        r'\label{tab:outbreak_detailed_metrics}',
        r'\footnotesize',
        r'\begin{tabular}{l|cccccc}',
        r'\toprule',
        r' & \multicolumn{3}{c}{\textbf{Lassa}} & \multicolumn{3}{c}{\textbf{Zika}} \\',
        r' & Precision & Recall & F1 & Precision & Recall & F1 \\',
        r'\midrule',
    ]

    def data_cols(r):
        return [_fmt(r.lassa_p), _fmt(r.lassa_r), _fmt(r.lassa_f1),
                _fmt(r.zika_p),  _fmt(r.zika_r),  _fmt(r.zika_f1)]

    for cat_name in CATEGORIES:
        cat_rows = df[df['metric'] == cat_name]
        if cat_rows.empty:
            continue
        lines.append(rf'\multicolumn{{7}}{{l}}{{\textbf{{{cat_name}}}}} \\')
        for _, r in cat_rows.iterrows():
            lines.append(_row_str([r['field']] + data_cols(r)))
        lines.append(r'\midrule')

    r = df[df['metric'] == 'Overall'].iloc[0]
    lines.append(_row_str([r'\textbf{Overall}'] + data_cols(r)))

    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table}']
    return '\n'.join(lines)



In [33]:
print(to_latex_detailed(det_df))

\begin{table}[h]
\centering
\caption{\textbf{Detailed precision, recall, and F1 for outbreak feature extraction by category.} Each row shows field-level performance within the four major epidemiological categories. Temporal and case burden features showed consistently high performance, while location-specific fields and epidemiological context features showed greater variability. Overall metrics represent the average across all 17 extraction fields.}
\label{tab:outbreak_detailed_metrics}
\footnotesize
\begin{tabular}{l|cccccc}
\toprule
 & \multicolumn{3}{c}{\textbf{Lassa}} & \multicolumn{3}{c}{\textbf{Zika}} \\
 & Precision & Recall & F1 & Precision & Recall & F1 \\
\midrule
\multicolumn{7}{l}{\textbf{Temporal Features}} \\
Start Year & 0.89 & 0.80 & 0.84 & 0.90 & 0.82 & 0.86 \\
Start Month & 0.78 & 0.78 & 0.78 & 0.80 & 0.80 & 0.80 \\
Start Day & 0.86 & 0.67 & 0.75 & 0.95 & 0.95 & 0.95 \\
End Month & 0.78 & 0.78 & 0.78 & 0.65 & 0.65 & 0.65 \\
End Day & 0.86 & 0.67 & 0.75 & 0.95 & 0.86 